# 💧 AIRA Kalkulator Ketahanan Air: Rule-Based Calculation & AI Recommendation dari Supabase Action Catalog

Selamat datang di **Hands-on Assignment** proyek AIRA! Notebook ini membangun dan menguji pipeline awal fitur **Kalkulator Ketahanan Air**, yang membantu rumah tangga memperkirakan kebutuhan air harian, cadangan air yang tersedia, dan estimasi durasi cadangan. Berdasarkan hasil perhitungan serta kondisi sumber air yang dipilih pengguna, sistem menentukan status dan memilih rekomendasi tindakan.

Pada pipeline ini, **perhitungan dan penetapan status dilakukan secara deterministik (rule-based)**. AI/LLM hanya menjadi lapisan opsional yang memilih dan mengurutkan tindakan dari katalog rekomendasi yang sudah ditetapkan, lalu menjelaskannya dengan bahasa yang mudah dipahami. AI tidak menghitung ulang angka, tidak mengubah status, dan tidak menjadi satu-satunya sumber rekomendasi.

### Alur Kerja (Workflow)
1. **Input Data Rumah Tangga**: lokasi opsional, jumlah anggota keluarga, sumber air utama, kondisi sumber air, serta cadangan air melalui volume aktual atau kapasitas tangki dan persentase isi. Kebutuhan MHM dapat dicatat secara opsional sebagai konteks rekomendasi.
2. **Validasi Input**: memeriksa kelengkapan, tipe data, rentang nilai, dan konsistensi data.
3. **Rule-Based Calculation**: menghitung kebutuhan air harian, cadangan air, dan estimasi durasi ketahanan.
4. **Klasifikasi Kondisi Air**: menetapkan status berdasarkan durasi dan kondisi sumber air, dengan aturan yang eksplisit.
5. **Recommendation Selection**: memilih tindakan dari katalog rekomendasi yang ditetapkan berdasarkan status dan kondisi sumber air.
6. **AI Recommendation Layer (Opsional)**: LLM memilih dan mengurutkan tindakan dari katalog yang ditetapkan serta memberi alasan singkat, tanpa menambahkan tindakan baru atau mengubah angka/status. Panggilan AI dilindungi retry, model cadangan, dan fallback: jika AI tetap gagal, rekomendasi rule-based tetap ditampilkan.
7. **Pengujian & Interpretasi**: menjalankan beberapa skenario, memeriksa batas status, dan mencatat keterbatasan.

> **Batasan penting:** angka 15 liter/orang/hari dan ambang status pada notebook ini adalah asumsi awal prototype untuk simulasi kebutuhan dasar, bukan standar kebutuhan normal semua rumah tangga atau standar resmi penetapan kekeringan. Hasil tidak membuktikan kualitas air aman diminum.

## 🎯 1. Problem Statement & Feature Description

### Latar belakang dan problem statement

Air adalah kebutuhan dasar untuk minum, memasak, kebersihan diri, dan sanitasi rumah tangga. Di wilayah rawan kekeringan, akses air sering terbatas, terutama saat kemarau berkepanjangan. Kondisi ini menambah beban perempuan, yang umumnya mengatur kebutuhan air dan kebersihan keluarga serta mencari sumber alternatif ketika persediaan menipis. Di sisi lain, informasi tentang kebutuhan air, pengelolaan cadangan, sanitasi, dan lokasi sumber air belum tentu mudah ditemukan dalam satu tempat.

**Masalah utamanya:** pengguna kesulitan memperkirakan **kebutuhan air harian dan berapa lama cadangan air dapat memenuhi kebutuhan keluarga**, serta membutuhkan **panduan tindakan yang perlu diprioritaskan** ketika persediaan mulai menipis. Pengguna juga membutuhkan informasi yang sederhana dan sesuai dengan kondisi rumah tangganya agar dapat mengambil keputusan dengan tepat.

**Solusi yang diuji pada notebook ini:** Kalkulator Ketahanan Air, salah satu fitur utama AIRA. Fitur ini menghitung kebutuhan, cadangan, dan durasi ketahanan air rumah tangga, menetapkan status kondisi air, lalu memberikan rekomendasi tindakan yang mudah dipahami.

**Pengguna sasaran:** perempuan dan keluarga, khususnya ibu rumah tangga di wilayah yang rentan kekeringan atau memiliki keterbatasan akses air. Kalkulator dapat dipakai langsung tanpa login.

### Fitur yang diuji
**Kalkulator Ketahanan Air** menghitung kebutuhan air harian, cadangan, dan estimasi durasi, lalu menetapkan **status kondisi air** dan menampilkan **rekomendasi tindakan** yang sesuai.

Alurnya: **input rumah tangga → hasil kalkulasi → status kondisi → rekomendasi → akses lanjutan ke Navigation Air atau AIRA Assistant.**

Sebagai gambaran: keluarga beranggotakan 4 orang membutuhkan sekitar 4 × 15 = 60 liter per hari. Jika cadangannya 600 liter, air diperkirakan bertahan 600 ÷ 60 = 10 hari, sehingga statusnya **Aman** (dengan asumsi tanpa kebutuhan MHM dan sumber air normal).

### Input: apa yang perlu diisi pengguna?
| Data | Wajib / Opsional | Pada pipeline ini |
|---|---|---|
| Lokasi atau wilayah | Opsional | Boleh dikosongkan; tidak dikirim ke AI |
| Jumlah anggota keluarga | Wajib | Bilangan bulat lebih dari 0 |
| Sumber air utama | Wajib | Wajib diisi (mis. sumur, PDAM, air hujan) |
| Kondisi sumber air | Wajib | Normal, Terbatas, Tidak berfungsi, atau Belum diketahui |
| Volume cadangan air | Wajib untuk menghitung durasi | Volume aktual (liter), atau kapasitas tangki × persentase isi (0–100%) |
| Kapasitas penampungan | Jika memakai penampungan | Wajib bila volume dihitung dari persentase; bila volume aktual diisi, tidak boleh melebihi kapasitas |
| Kebutuhan air harian | Ditetapkan sistem | 15 liter/orang/hari sebagai asumsi awal prototype |
| Kebutuhan MHM | Opsional | `KEBUTUHAN_MHM` dan `JUMLAH_ANGGOTA_MHM` (tidak boleh melebihi jumlah anggota keluarga) |
| Luas atap dan curah hujan | Opsional | Belum dimasukkan; panen air hujan adalah pengembangan terpisah |

### Bagaimana hasilnya dihitung?
- **Kebutuhan harian** = jumlah anggota keluarga × kebutuhan air per orang per hari.
- **Cadangan air** = volume aktual, atau kapasitas tangki × (persentase isi ÷ 100).
- **Durasi cadangan** = cadangan ÷ kebutuhan harian. Jika kebutuhan harian nol atau tidak valid, pembagian tidak dilakukan. Status ditentukan dari angka durasi yang belum dibulatkan, agar tidak bergeser karena pembulatan tampilan.

Dari durasi itulah status ditetapkan:

| Status | Ambang durasi | Aksi utama untuk pengguna |
|---|---|---|
| Aman / Relatif Mencukupi | Lebih dari 3 hari | Tanya AIRA Assistant |
| Waspada | Lebih dari 1 hari hingga 3 hari (tepat 1 dan 3 hari termasuk Waspada) | Tanya AIRA Assistant |
| Mode Krisis | Kurang dari 1 hari | Cari Titik Air di Navigation Air |
| Belum Dapat Ditentukan | Data tidak lengkap, tidak valid, atau belum mencukupi | Periksa Kembali Data |

Jika data belum cukup, sistem tidak langsung menyimpulkan Aman. Pengguna diminta melengkapi atau memperbaiki datanya terlebih dahulu.

### Keputusan tim di luar rumus dasar rancangan produk
Ada dua aturan pada pipeline ini yang melampaui rumus dasar dan perlu dievaluasi lebih lanjut:
- **Tambahan kebutuhan MHM.** Rumus dasar hanya 15 liter × jumlah anggota, sedangkan perhitungan berbasis kategori penggunaan (termasuk MHM) dirancang untuk dikembangkan kemudian. Agar kebutuhan menstruasi tidak terabaikan, pipeline menambahkan **5 liter/orang/hari** untuk setiap anggota yang memerlukan MHM. Angka ini adalah alokasi operasional prototype, bukan standar universal.
- **Kondisi sumber air.** Ambang status pada dasarnya berbasis durasi. Pipeline menambahkan penyesuaian yang berhati-hati: *Tidak berfungsi* → Mode Krisis; *Terbatas* dengan durasi lebih dari 3 hari → Waspada; *Belum diketahui* → Belum Dapat Ditentukan. Aturan ini masih keputusan prototype yang perlu divalidasi.

### Output: apa yang didapat pengguna?
Pengguna memperoleh gambaran lengkap: kebutuhan air harian, perkiraan volume cadangan, estimasi durasi, status kondisi air, **alasan** di balik status tersebut, rekomendasi tindakan, dan waktu perhitungan. Semuanya disampaikan sebagai **estimasi berdasarkan data yang dimasukkan**, bukan pengukuran langsung.

### Di mana peran AI, dan apa batasannya?
- **Angka dan status dihitung oleh kode** dengan rumus dan ambang yang konsisten serta dapat diuji.
- **AI (Gemini) membantu menyusun rekomendasi**: memilih dan mengurutkan tindakan dari katalog yang sudah disiapkan, lalu memberi alasan singkat untuk tiap pilihan.
- **AI tidak boleh** mengubah hasil perhitungan, mengarang angka, atau menentukan dan mengubah status, termasuk Mode Krisis.
- **Jika AI tidak tersedia**, hasil perhitungan dan rekomendasi rule-based tetap ditampilkan, sehingga pengguna tidak kehilangan hasil.
- **Hasil hanyalah estimasi.** Kebutuhan 15 liter/orang/hari dan ambang status adalah asumsi awal prototype, bukan standar resmi. Volume air juga tidak menunjukkan bahwa air aman diminum.

### Peran dalam final project
Kalkulator adalah pintu masuk alur AIRA, dan statusnya menunjukkan langkah berikutnya:
- **Aman:** jaga cadangan dan gunakan air dengan bijak.
- **Waspada:** berhemat dan mulai menyiapkan sumber alternatif.
- **Mode Krisis:** prioritaskan kebutuhan esensial dan cari titik air melalui **Navigation Air**.
- **Belum Dapat Ditentukan:** lengkapi atau perbaiki data, lalu hitung ulang.

Hasil terstruktur kalkulator juga menjadi bahan bagi **AIRA Assistant** untuk menjelaskan hasil tanpa menghitung ulang. Navigation Air dan AIRA Assistant tetap dapat dibuka dari menu utama pada semua status. Penyimpanan riwayat setelah login berada di luar cakupan pipeline ini.

## 📦 2. Setup dan Konstanta Prototype

Notebook ini dapat menjalankan perhitungan, rekomendasi rule-based, dan pengujian **tanpa API key Gemini**. Katalog tindakan tetap dibutuhkan: dibaca dari tabel Supabase, atau dari file CSV cadangan bila Supabase tidak dapat diakses. Integrasi Gemini bersifat opsional. Untuk mengaktifkannya, pasang `google-genai` dan simpan API key pada environment variable `GEMINI_API_KEY` atau Colab Secrets. Jangan menuliskan API key langsung di notebook atau mengunggahnya ke GitHub.

In [1]:
!pip install -q google-genai supabase pandas

In [2]:
import os
import json
import math
import time
import random
import pandas as pd
from datetime import datetime, timezone

# display() tersedia di Jupyter/Colab; di luar itu pakai print.
try:
    from IPython.display import display
except ImportError:
    display = print

# Colab Secrets bersifat opsional: di luar Colab (lokal/VS Code) notebook tetap
# bisa dijalankan dengan environment variable.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# SDK Gemini bersifat opsional: tanpa SDK, pipeline memakai rule-based fallback.
try:
    from google import genai
except ImportError:
    genai = None

# Klien Supabase bersifat opsional: tanpa klien, katalog dibaca dari CSV cadangan.
try:
    from supabase import create_client
except ImportError:
    create_client = None

print("✅ Library siap digunakan.")

✅ Library siap digunakan.


### 🔌 Hubungkan Modul Kalkulator dan Katalog Tindakan AIRA dari Supabase

Dua sel berikut dijalankan **berurutan**:

1. **Modul bersama `aira_calculator_shared.py`.** File ini adalah satu-satunya sumber rumus dan klasifikasi status Kalkulator Air, dan juga dipakai notebook AIRA Assistant. Jika file belum ada di direktori kerja, sel akan meminta Anda mengunggahnya (di Colab). Konstanta status dari modul ini dipakai untuk memeriksa isi katalog, sehingga sel ini harus berjalan lebih dulu.
2. **Katalog tindakan dari Supabase.** Katalog dibaca dari tabel Supabase (default `katalog_tindakan`, dapat diubah lewat environment variable `SUPABASE_TABLE_KATALOG`), lalu divalidasi dan dinormalisasi sebelum dipakai.

**Persiapan Colab Secrets:**
- `SUPABASE_URL`: Project URL Supabase.
- `SUPABASE_SERVICE_ROLE_KEY`: service role key untuk notebook privat (nama lama `SUPABASE_ROLE_KEY` tetap dikenali, jadi nama secret sama dengan notebook AIRA Assistant).
- `GEMINI_API_KEY`: opsional, untuk mengaktifkan Gemini.

**Aturan isi katalog yang dipakai kode:**
- Kolom `prioritas_*` **boleh kosong**. Sel kosong berarti tindakan itu bukan kandidat untuk status tersebut (misalnya tindakan khusus Mode Krisis tidak punya `prioritas_aman`). Yang dianggap salah adalah nilai yang terisi tetapi bukan angka.
- `status_berlaku` dan `kondisi_sumber_berlaku` ditulis dipisah titik koma (`;`).
- Aturan `wajib` yang dikenal: `Ya`, `Tidak`, `Jika MHM aktif`, `Jika Mode Krisis dan fitur tersedia`, `Ya jika data tidak terverifikasi`, dan `Jika ada dugaan kontaminasi`.

**Cadangan tanpa Supabase (opsional):** jika Supabase tidak dapat dihubungi dan file `AIRA_Katalog_Tindakan_Final_REVISED.csv` ada di direktori kerja, katalog dibaca dari CSV dan notebook menampilkan peringatan jelas tentang sumber yang dipakai. Ini memungkinkan mentor atau reviewer menjalankan notebook tanpa akses ke database privat. Matikan dengan `IZINKAN_FALLBACK_CSV = False`.

Jangan pernah menaruh service role key di frontend website atau membagikan notebook yang berisi kredensial.

In [3]:
# MODUL KALKULATOR BERSAMA (sumber tunggal rumus dan klasifikasi status)
import sys
import importlib

NAMA_MODUL_BERSAMA = "aira_calculator_shared.py"

if not os.path.exists(NAMA_MODUL_BERSAMA):
    try:
        from google.colab import files
        print(f"{NAMA_MODUL_BERSAMA} belum ada. Unggah file tersebut sekarang:")
        files.upload()
    except ImportError:
        pass

if not os.path.exists(NAMA_MODUL_BERSAMA):
    raise FileNotFoundError(
        f"{NAMA_MODUL_BERSAMA} tidak ditemukan di {os.getcwd()}. "
        "Letakkan file itu di folder yang sama dengan notebook (atau unggah di Colab), "
        "lalu jalankan ulang sel ini. Pastikan nama file tidak berubah, misalnya menjadi "
        "'aira_calculator_shared (1).py'."
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import aira_calculator_shared
importlib.reload(aira_calculator_shared)
from aira_calculator_shared import (
    KEBUTUHAN_PER_ORANG_LITER_HARI,
    TAMBAHAN_AIR_MHM_PER_ORANG_LITER_HARI,
    BATAS_KRISIS_HARI, BATAS_WASPADA_HARI,
    STATUS_AMAN, STATUS_WASPADA, STATUS_KRISIS, STATUS_TIDAK_DITENTUKAN,
)
print("Konstanta kalkulator bersama siap.")

aira_calculator_shared.py belum ada. Unggah file tersebut sekarang:


Saving aira_calculator_shared.py to aira_calculator_shared.py
Fungsi kalkulator siap.
Fungsi kalkulator siap.
Konstanta kalkulator bersama siap.


In [4]:
# KONEKSI SUPABASE, PEMUATAN, VALIDASI, DAN NORMALISASI KATALOG TINDAKAN AIRA
# Sumber utama: tabel Supabase. Cadangan (opsional): CSV lokal untuk pihak tanpa akses Supabase.

SUPABASE_TABLE_KATALOG = os.getenv("SUPABASE_TABLE_KATALOG", "katalog_tindakan")
CSV_KATALOG_CADANGAN = "AIRA_Katalog_Tindakan_Final_REVISED.csv"
IZINKAN_FALLBACK_CSV = True

KOLOM_WAJIB = [
    "action_id", "kategori", "nama_tindakan", "teks_tindakan", "konteks_pemicu",
    "status_berlaku", "kondisi_sumber_berlaku", "syarat_mhm", "wajib",
    "prioritas_aman", "prioritas_waspada", "prioritas_krisis", "prioritas_belum_ditentukan",
    "cta", "catatan_keamanan", "dasar_rujukan", "aktif",
]

# Status -> kolom prioritas. Nilai status diambil dari modul bersama agar tidak berbeda.
KOLOM_PRIORITAS = {
    STATUS_AMAN: "prioritas_aman",
    STATUS_WASPADA: "prioritas_waspada",
    STATUS_KRISIS: "prioritas_krisis",
    STATUS_TIDAK_DITENTUKAN: "prioritas_belum_ditentukan",
}
KONDISI_DIKENAL = ["Semua kondisi", "Normal", "Terbatas", "Tidak berfungsi", "Belum diketahui"]
ATURAN_WAJIB_DIKENAL = {
    "ya", "tidak", "jika mhm aktif", "jika mode krisis dan fitur tersedia",
    "ya jika data tidak terverifikasi", "jika ada dugaan kontaminasi",
}


# ---------- Pembersih nilai (tahan terhadap None, NaN, boolean, dan list dari Supabase) ----------
def _kosong(nilai):
    if nilai is None:
        return True
    if isinstance(nilai, str):
        return nilai.strip() == ""          # sel kosong dari CSV/Supabase kadang berupa string kosong
    if isinstance(nilai, (list, tuple, set, dict)):
        return len(nilai) == 0
    try:
        return bool(pd.isna(nilai))
    except (TypeError, ValueError):
        return False

def _teks(nilai):
    if _kosong(nilai):
        return ""
    if isinstance(nilai, (list, tuple, set)):
        return "; ".join(str(x).strip() for x in nilai)
    return str(nilai).strip()

def _kunci(nilai):
    """Bentuk pembanding: huruf kecil dan spasi dirapikan."""
    return " ".join(_teks(nilai).casefold().split())

def _daftar(nilai):
    """Ubah 'A; B' atau ['A', 'B'] menjadi list ['A', 'B']."""
    if isinstance(nilai, (list, tuple, set)):
        butir = [str(x) for x in nilai]
    else:
        butir = _teks(nilai).split(";")
    return [b.strip() for b in butir if b.strip()]

def _ya_tidak(nilai):
    k = _kunci(nilai)
    if k in {"ya", "yes", "true", "1", "aktif"}:
        return "ya"
    if k in {"tidak", "no", "false", "0", "nonaktif", ""}:
        return "tidak"
    return k

def _angka(nilai):
    """None jika kosong; ValueError jika terisi tetapi bukan angka."""
    if _kosong(nilai):
        return None
    return float(str(nilai).strip().replace(",", "."))


# ---------- Pengambilan data ----------
def ambil_kredensial(*nama_secret):
    """Cari nilai dari environment variable lalu Colab Secrets; pakai nama pertama yang tersedia."""
    for nama in nama_secret:
        nilai = os.getenv(nama)
        if not nilai and userdata is not None:
            try:
                nilai = userdata.get(nama)
            except Exception:  # secret belum dibuat atau akses notebook belum diizinkan
                nilai = None
        if nilai:
            return nilai.strip()
    return None

def ambil_semua_baris_supabase(client, nama_tabel, ukuran_halaman=500):
    """Ambil seluruh baris secara bertahap. Diurutkan agar halaman tidak tumpang tindih."""
    semua, offset = [], 0
    while True:
        batch = (
            client.table(nama_tabel).select("*").order("action_id")
            .range(offset, offset + ukuran_halaman - 1).execute().data
        ) or []
        semua.extend(batch)
        if len(batch) < ukuran_halaman:
            break
        offset += ukuran_halaman
    return semua

def muat_katalog_mentah():
    """Kembalikan (baris, sumber). Utamakan Supabase; CSV hanya sebagai cadangan yang dinyatakan jelas."""
    url = ambil_kredensial("SUPABASE_URL")
    kunci = ambil_kredensial("SUPABASE_SERVICE_ROLE_KEY", "SUPABASE_ROLE_KEY")
    sebab = None

    if create_client is None:
        sebab = "paket supabase belum terpasang"
    elif not (url and kunci):
        sebab = "SUPABASE_URL atau SUPABASE_SERVICE_ROLE_KEY belum tersedia di Colab Secrets/environment"
    else:
        try:
            client = create_client(url, kunci)
            baris = ambil_semua_baris_supabase(client, SUPABASE_TABLE_KATALOG)
            if not baris:
                raise ValueError(
                    f"tabel '{SUPABASE_TABLE_KATALOG}' kosong atau tidak dapat dibaca "
                    "(periksa nama tabel dan hak akses)"
                )
            print(f"Koneksi Supabase berhasil. Baris diambil: {len(baris)}")
            return baris, f"Supabase (tabel {SUPABASE_TABLE_KATALOG})"
        except Exception as e:
            sebab = f"{type(e).__name__}: {e}"

    if IZINKAN_FALLBACK_CSV and os.path.exists(CSV_KATALOG_CADANGAN):
        print(f"⚠️ Supabase tidak dipakai ({sebab}).")
        print(f"⚠️ MODE CADANGAN: katalog dibaca dari file lokal {CSV_KATALOG_CADANGAN}. "
              "Isinya bisa berbeda dari tabel Supabase.")
        df_csv = pd.read_csv(CSV_KATALOG_CADANGAN, dtype=str)
        baris = [
            {k: (None if pd.isna(v) else v) for k, v in rec.items()}
            for rec in df_csv.to_dict("records")
        ]
        return baris, f"CSV cadangan ({CSV_KATALOG_CADANGAN})"

    raise RuntimeError(
        f"Gagal memuat katalog dari Supabase: {sebab}. "
        f"Tidak ada file cadangan {CSV_KATALOG_CADANGAN} di {os.getcwd()}."
    )


# ---------- Validasi dan normalisasi ----------
def normalisasi_katalog(baris):
    """Validasi seluruh baris katalog. Kesalahan struktur dihentikan; kejanggalan data hanya diberi peringatan.

    Catatan: sel prioritas yang kosong BUKAN kesalahan. Kosong berarti tindakan itu
    tidak menjadi kandidat untuk status tersebut.
    """
    if not baris:
        raise ValueError("Katalog kosong.")

    df = pd.DataFrame(baris)
    hilang = [k for k in KOLOM_WAJIB if k not in df.columns]
    if hilang:
        raise ValueError("Kolom berikut belum tersedia di katalog: " + ", ".join(hilang))

    status_dikenal = {_kunci(s): s for s in KOLOM_PRIORITAS}
    kondisi_dikenal = {_kunci(k) for k in KONDISI_DIKENAL}
    kesalahan, peringatan, records, terlihat = [], [], [], set()

    for no, row in enumerate(baris, start=1):
        aid = _teks(row.get("action_id"))
        if not aid:
            kesalahan.append(f"baris {no}: action_id kosong")
            continue
        if aid in terlihat:
            kesalahan.append(f"action_id duplikat: {aid}")
            continue
        terlihat.add(aid)

        if not _teks(row.get("teks_tindakan")):
            kesalahan.append(f"{aid}: teks_tindakan kosong")

        status_row = []
        for s in _daftar(row.get("status_berlaku")):
            if _kunci(s) in status_dikenal:
                status_row.append(status_dikenal[_kunci(s)])
            else:
                kesalahan.append(f"{aid}: status tidak dikenal '{s}'")
        if not status_row:
            peringatan.append(f"{aid}: status_berlaku kosong (tidak pernah dipilih)")

        kondisi_row = {_kunci(k) for k in _daftar(row.get("kondisi_sumber_berlaku"))}
        if not kondisi_row:
            kesalahan.append(f"{aid}: kondisi_sumber_berlaku kosong")
        for k in sorted(kondisi_row - kondisi_dikenal):
            kesalahan.append(f"{aid}: kondisi sumber tidak dikenal '{k}'")

        wajib = _ya_tidak(row.get("wajib"))
        if wajib not in ATURAN_WAJIB_DIKENAL:
            peringatan.append(f"{aid}: aturan wajib tidak dikenal '{_teks(row.get('wajib'))}' (dianggap tidak wajib)")

        prioritas = {}
        for status, kolom in KOLOM_PRIORITAS.items():
            try:
                nilai = _angka(row.get(kolom))
            except ValueError:
                kesalahan.append(f"{aid}: {kolom} bukan angka ('{row.get(kolom)}')")
                nilai = None
            if nilai is not None and nilai <= 0:
                peringatan.append(f"{aid}: {kolom} bernilai {nilai:g} (seharusnya lebih dari 0)")
            prioritas[status] = nilai
            if status in status_row and nilai is None and wajib == "tidak":
                peringatan.append(f"{aid}: berlaku untuk '{status}' tetapi {kolom} kosong (tidak akan terpilih)")
            if status not in status_row and nilai is not None:
                peringatan.append(f"{aid}: {kolom} terisi tetapi '{status}' tidak ada di status_berlaku (diabaikan)")

        records.append({
            "id": aid,
            "kategori": _teks(row.get("kategori")),
            "nama": _teks(row.get("nama_tindakan")),
            "tindakan": _teks(row.get("teks_tindakan")),
            "konteks": _teks(row.get("konteks_pemicu")),
            "status": set(status_row),
            "kondisi": kondisi_row,
            "syarat_mhm": _ya_tidak(row.get("syarat_mhm")) == "ya",
            "wajib": wajib,
            "prioritas": prioritas,
            "cta": _teks(row.get("cta")),
            "catatan_keamanan": _teks(row.get("catatan_keamanan")),
            "dasar_rujukan": _teks(row.get("dasar_rujukan")),
            "aktif": _ya_tidak(row.get("aktif")) == "ya",
        })

    if kesalahan:
        tampil = "\n- ".join(kesalahan[:20])
        lebih = f"\n(+{len(kesalahan) - 20} masalah lain)" if len(kesalahan) > 20 else ""
        raise ValueError("Katalog tidak valid:\n- " + tampil + lebih)

    # Salinan tabel untuk tampilan dan pengujian: kolom prioritas dijadikan angka (kosong = NaN).
    for kolom in KOLOM_PRIORITAS.values():
        df[kolom] = pd.to_numeric(df[kolom].map(lambda v: None if _kosong(v) else str(v).replace(",", ".")), errors="coerce")
    return records, df, peringatan


baris_katalog, SUMBER_KATALOG_AKTIF = muat_katalog_mentah()
KATALOG_RECORDS, katalog_tindakan_df, peringatan_katalog = normalisasi_katalog(baris_katalog)

print("\n" + "=" * 55)
print("KATALOG TINDAKAN AIRA BERHASIL DIMUAT")
print("=" * 55)
print("Sumber katalog   :", SUMBER_KATALOG_AKTIF)
print("Jumlah tindakan  :", len(KATALOG_RECORDS), f"(aktif: {sum(r['aktif'] for r in KATALOG_RECORDS)})")
print("Jumlah kolom     :", len(katalog_tindakan_df.columns))

ringkasan_status = pd.DataFrame([
    {
        "Status": status,
        "Tindakan berlaku": sum(1 for r in KATALOG_RECORDS if r["aktif"] and status in r["status"]),
        "Kandidat berprioritas": sum(
            1 for r in KATALOG_RECORDS
            if r["aktif"] and status in r["status"] and r["prioritas"].get(status) is not None
        ),
    }
    for status in KOLOM_PRIORITAS
])
print("\nKetersediaan tindakan per status (sebelum filter kondisi sumber dan MHM):")
display(ringkasan_status)

if peringatan_katalog:
    print(f"\n⚠️ {len(peringatan_katalog)} peringatan kualitas data katalog (tidak menghentikan notebook):")
    for p in peringatan_katalog[:10]:
        print("  -", p)
else:
    print("\nTidak ada peringatan kualitas data katalog.")

print("\nContoh data katalog:")
display(katalog_tindakan_df.head())

Koneksi Supabase berhasil. Baris diambil: 77

KATALOG TINDAKAN AIRA BERHASIL DIMUAT
Sumber katalog   : Supabase (tabel katalog_tindakan)
Jumlah tindakan  : 77 (aktif: 77)
Jumlah kolom     : 17

Ketersediaan tindakan per status (sebelum filter kondisi sumber dan MHM):


,Status,Tindakan berlaku,Kandidat berprioritas
0,Aman / Relatif Mencukupi,42,42
1,Waspada,68,68
2,Mode Krisis,60,60
3,Belum Dapat Ditentukan,8,8



Tidak ada peringatan kualitas data katalog.

Contoh data katalog:


,action_id,kategori,nama_tindakan,teks_tindakan,konteks_pemicu,status_berlaku,kondisi_sumber_berlaku,syarat_mhm,wajib,prioritas_aman,prioritas_waspada,prioritas_krisis,prioritas_belum_ditentukan,cta,catatan_keamanan,dasar_rujukan,aktif
0,bagi_tugas_pemantauan,Keluarga,Bagi tugas pemantauan cadangan dalam keluarga,"Jika sesuai, sepakati siapa yang memperbarui c...",Keluarga ingin membuat rencana pengelolaan cad...,Aman / Relatif Mencukupi; Waspada,Semua kondisi,Tidak,Tidak,9.0,9.0,NaN,NaN,Lihat Rekomendasi,Sesuaikan pembagian tugas dengan kemampuan dan...,WHO/UNICEF/CDC; adaptasi AIRA,Ya
1,beri_tahu_kebutuhan_khusus,Keluarga,Sampaikan kebutuhan khusus saat mencari bantuan,"Saat meminta bantuan, sampaikan kebutuhan akse...",Ada anggota keluarga dengan kebutuhan akses at...,Waspada; Mode Krisis,Semua kondisi,Tidak,Tidak,NaN,6.0,5.0,NaN,Lihat Rekomendasi,Jangan meminta atau menyimpan data kesehatan s...,WHO/UNICEF/CDC; adaptasi AIRA,Ya
2,buka_navigation_air,Akses sumber,Tinjau titik air melalui Navigation Air,Gunakan Navigation Air untuk meninjau kandidat...,Pengguna perlu mencari titik air alternatif,Mode Krisis; Waspada,Semua kondisi,Tidak,Jika Mode Krisis dan fitur tersedia,NaN,3.0,2.0,NaN,Cari Titik Air,Titik yang belum diverifikasi harus ditandai s...,WHO/UNICEF/CDC; adaptasi AIRA,Ya
3,cari_sumber_alternatif,Akses sumber,Mulai mencari sumber air alternatif,Cari pilihan sumber air alternatif dan konfirm...,Sumber utama terbatas atau tidak berfungsi,Waspada; Mode Krisis,Terbatas; Tidak berfungsi; Belum diketahui,Tidak,Tidak,NaN,2.0,2.0,NaN,Buka Navigation Air,Jangan anggap titik tersedia atau air layak mi...,WHO/UNICEF/CDC; adaptasi AIRA,Ya
4,catat_pola_pemakaian,Monitoring,Catat pola pemakaian air,Catat perkiraan penggunaan air rumah tangga se...,Pengguna ingin membuat estimasi lebih sesuai d...,Aman / Relatif Mencukupi; Waspada,Semua kondisi,Tidak,Tidak,4.0,4.0,NaN,NaN,Perbarui Kalkulator,Pencatatan tidak perlu mengumpulkan data priba...,WHO/UNICEF/CDC; adaptasi AIRA,Ya


## 🧮 3. Fungsi Validasi, Perhitungan, dan Klasifikasi

### Aturan yang digunakan

- **Kebutuhan dasar harian** = jumlah seluruh anggota keluarga × asumsi kebutuhan air per orang per hari.
- **Tambahan kebutuhan air MHM** = jumlah anggota keluarga yang membutuhkan MHM × 5 liter/orang/hari. Jika tidak ada anggota yang membutuhkan MHM, tambahan kebutuhan air MHM adalah 0 liter/hari.
- **Total kebutuhan air harian** = kebutuhan dasar harian + tambahan kebutuhan air MHM.
- **Cadangan air** = volume aktual jika diketahui; jika tidak, kapasitas tangki × persentase isi / 100.
- **Durasi** = cadangan air / total kebutuhan air harian.
- Durasi **kurang dari 1 hari** → Mode Krisis; **1 sampai dengan 3 hari** → Waspada; **lebih dari 3 hari** → Aman / Relatif Mencukupi.
- Kondisi sumber **Tidak berfungsi** menjadi pemicu Mode Krisis; kondisi **Terbatas** mencegah status Aman dan minimal menghasilkan Waspada. Jika kondisi sumber **Belum diketahui**, status keseluruhan belum dapat ditentukan.
- Data tidak valid atau wajib yang tidak lengkap menghasilkan Belum Dapat Ditentukan, bukan Aman.
- Jumlah anggota yang membutuhkan MHM tidak boleh melebihi jumlah seluruh anggota keluarga.




**Catatan acuan kebutuhan air:**

Perhitungan kebutuhan air dasar AIRA menggunakan acuan minimum **15 liter/orang/hari** berdasarkan *Sphere Handbook*, yang menetapkan jumlah tersebut sebagai alokasi minimum untuk kebutuhan minum, memasak, serta kebersihan pribadi dan rumah tangga dalam respons kemanusiaan. Angka ini merupakan acuan untuk konteks darurat, bukan batas maksimum atau standar kebutuhan ideal untuk semua kondisi.




Untuk mendukung kebutuhan Manajemen Kebersihan Menstruasi (MHM), AIRA mengalokasikan tambahan **5 liter/orang/hari bagi setiap anggota keluarga yang memerlukan MHM**. Alokasi ini mempertimbangkan kebutuhan akses air untuk membersihkan diri dan tangan serta mengelola bahan menstruasi secara higienis, sebagaimana dijelaskan dalam panduan kesehatan dan kebersihan menstruasi UNICEF. Namun, angka tambahan 5 liter/orang/hari merupakan **alokasi operasional yang digunakan AIRA**, bukan angka minimum universal yang secara khusus ditetapkan oleh Sphere atau UNICEF. Besaran ini perlu dievaluasi berdasarkan praktik MHM dan kondisi penggunaan air di wilayah sasaran.



Aturan klasifikasi kondisi sumber air dan status ketahanan air merupakan logika awal kalkulator AIRA. Aturan tersebut perlu diuji dan dievaluasi bersama pihak yang memahami kebutuhan air, sanitasi, serta konteks lokal sebelum digunakan sebagai dasar rekomendasi di kondisi nyata.




>_Daftar Referensi_:
> - https://interactive.spherestandards.org/
> (Sphere Standards — Water Supply).
> - https://www.unicef.org/documents/guidance-menstrual-health-and-hygiene
> (UNICEF — Guidance on Menstrual Health and Hygiene).

In [5]:
# Gunakan implementasi kalkulator tunggal yang juga dipakai AIRA Assistant.
from aira_calculator_shared import (
    _is_int, _is_angka, klasifikasikan_status, hitung_ketahanan_air
)
print("Fungsi Kalkulator Air diimpor dari aira_calculator_shared.py.")


Fungsi Kalkulator Air diimpor dari aira_calculator_shared.py.


## 🧾 4. Input Simulasi (disesuaikan dengan form Figma)

Ubah nilai pada sel ini untuk mencoba skenario lain. Lokasi bersifat opsional. Untuk volume cadangan, isi **salah satu** metode: `VOLUME_CADANGAN_AKTUAL_LITER`, atau `KAPASITAS_TANGKI_LITER` beserta `PERSENTASE_ISI_TANGKI`.

Kebutuhan MHM dicatat melalui dua input: `KEBUTUHAN_MHM` untuk menunjukkan apakah ada anggota keluarga yang memerlukan MHM, dan `JUMLAH_ANGGOTA_MHM` untuk mencatat jumlah anggota tersebut. Jumlah anggota yang memerlukan MHM tidak boleh melebihi jumlah seluruh anggota keluarga.

Kalkulator menghitung kebutuhan air dasar sebesar 15 liter/orang/hari dan menambahkan alokasi 5 liter/orang/hari untuk setiap anggota keluarga yang memerlukan MHM. Dengan demikian, tambahan kebutuhan MHM ikut memengaruhi estimasi lama cadangan air dapat mencukupi.

**Catatan:** Acuan 15 liter/orang/hari digunakan untuk kebutuhan air dasar dalam konteks darurat. Tambahan 5 liter/orang/hari untuk MHM merupakan alokasi operasional AIRA yang perlu dievaluasi berdasarkan panduan dan kondisi lokal, bukan standar minimum universal yang ditetapkan secara khusus oleh Sphere atau UNICEF.

In [6]:
# Contoh simulasi — ubah sesuai skenario uji yang ingin dijalankan.
LOKASI = "Rongkop, Gunungkidul"
JUMLAH_ANGGOTA = 4
SUMBER_AIR_UTAMA = "Sumur"
KONDISI_SUMBER_AIR = "Terbatas"  # Normal / Terbatas / Tidak berfungsi / Belum diketahui

# Pilih satu metode cadangan:
VOLUME_CADANGAN_AKTUAL_LITER = None  # Contoh: 450. Isi None jika memakai kapasitas × persentase.
KAPASITAS_TANGKI_LITER = 1000
PERSENTASE_ISI_TANGKI = 45

KEBUTUHAN_MHM = True
JUMLAH_ANGGOTA_MHM = 2  # Jumlah anggota keluarga yang memerlukan MHM.

hasil_kalkulasi = hitung_ketahanan_air(
    jumlah_anggota=JUMLAH_ANGGOTA,
    sumber_air_utama=SUMBER_AIR_UTAMA,
    kondisi_sumber_air=KONDISI_SUMBER_AIR,
    volume_cadangan_liter=VOLUME_CADANGAN_AKTUAL_LITER,
    kapasitas_tangki_liter=KAPASITAS_TANGKI_LITER,
    persentase_isi=PERSENTASE_ISI_TANGKI,
    lokasi=LOKASI,
    kebutuhan_mhm=KEBUTUHAN_MHM,
    jumlah_anggota_mhm=JUMLAH_ANGGOTA_MHM,
)

print(json.dumps(hasil_kalkulasi, ensure_ascii=False, indent=2))

{
  "valid": true,
  "errors": [],
  "lokasi": "Rongkop, Gunungkidul",
  "jumlah_anggota": 4,
  "sumber_air_utama": "Sumur",
  "kondisi_sumber_air": "Terbatas",
  "kebutuhan_mhm": true,
  "jumlah_anggota_mhm": 2,
  "kebutuhan_per_orang_liter_hari": 15.0,
  "tambahan_air_mhm_per_orang_liter_hari": 5.0,
  "kebutuhan_dasar_liter": 60.0,
  "tambahan_air_mhm_liter": 10.0,
  "kebutuhan_harian_liter": 70.0,
  "cadangan_air_liter": 450.0,
  "durasi_hari": 6.43,
  "status_durasi": "Aman / Relatif Mencukupi",
  "status": "Waspada",
  "alasan_status": "Cadangan tersimpan diperkirakan cukup lebih dari 3 hari, tetapi sumber air utama sedang terbatas.",
  "metode_volume": "kapasitas × persentase",
  "waktu_perhitungan_utc": "2026-10-30T16:40:48.159323+00:00"
}


## 🧭 5. Recommendation System Berbasis Aturan

Rekomendasi dipilih oleh kode berdasarkan status hasil kalkulator dan kondisi sumber air. Ini membuat perilaku sistem konsisten serta dapat diuji. AI tidak menentukan status dan tidak menjadi sumber kebenaran angka.

- **Aman / Relatif Mencukupi:** pertahankan cadangan, pantau pemakaian, dan rencanakan pengisian ulang.
- **Waspada:** hemat air untuk penggunaan non-esensial, pantau cadangan, dan mulai mencari informasi sumber alternatif.
- **Mode Krisis:** prioritaskan kebutuhan dasar, hubungi pengelola/otoritas lokal atau sumber bantuan, dan buka Navigation Air untuk memeriksa titik alternatif. Jangan menganggap titik alternatif pasti tersedia tanpa konfirmasi.
- **Belum Dapat Ditentukan:** periksa data yang belum diketahui/invalid dan hitung ulang.

Panduan sanitasi/MHM tidak boleh mendorong pengguna mengabaikan kebersihan penting. Air yang sudah tercampur bahan kimia atau limbah tidak boleh disarankan untuk minum, memasak, atau kebersihan tubuh.

Semua teks tindakan disimpan **satu kali** di tabel Supabase `katalog_tindakan` (dimuat ke `KATALOG_RECORDS` pada Bagian 2). Rule-based memilih ID tindakan per status, dan AI (Bagian 6) memilih dari katalog yang sama, sehingga teks tidak ganda dan AI tidak dapat menciptakan tindakan baru.

**Cara katalog dipakai oleh kode:**
1. Hanya tindakan **aktif** yang dipertimbangkan.
2. Tindakan harus berlaku untuk **status** hasil kalkulator (`status_berlaku`) dan **kondisi sumber air** pengguna (`kondisi_sumber_berlaku`; `Semua kondisi` berlaku untuk semua).
3. Tindakan bersyarat MHM (`syarat_mhm = Ya`) hanya muncul jika ada anggota keluarga yang memerlukan MHM.
4. Tindakan harus punya **prioritas** untuk status itu. Sel prioritas yang kosong berarti bukan kandidat biasa untuk status tersebut.
5. Tindakan **wajib** (kolom `wajib`) selalu disertakan bila aturannya terpenuhi, apa pun prioritasnya. Urutan akhir: wajib dahulu, lalu prioritas terkecil.

In [7]:
# Recommendation System memakai katalog tindakan dari Supabase (KATALOG_RECORDS), bukan daftar hard-coded.
# Python tetap menjadi pengendali filter, tindakan wajib, dan prioritas dasar.

# ID tindakan yang dirujuk langsung oleh aturan sistem. Harus ada di katalog (diperiksa di sel validasi).
ID_NAVIGATION_AIR = "buka_navigation_air"
ID_PERIKSA_DATA = "periksa_data_kalkulator"
ID_MHM_UTAMA = "kebersihan_mhm"
ID_KONFIRMASI_KONDISI = "konfirmasi_kondisi_sumber"
ID_SUMBER_ALTERNATIF = "cari_sumber_alternatif"

# Aturan wajib "Jika Mode Krisis dan fitur tersedia": ubah ke False jika Navigation Air belum aktif.
FITUR_NAVIGATION_AIR_TERSEDIA = True

BATAS_REKOMENDASI_RULE = 5

def _mhm_aktif(data):
    jumlah = data.get("jumlah_anggota_mhm")
    return (
        data.get("kebutuhan_mhm") is True
        and isinstance(jumlah, int)
        and not isinstance(jumlah, bool)
        and jumlah > 0
    )

def _data_tidak_terverifikasi(data):
    """Data dianggap belum terverifikasi jika input tidak valid, status belum dapat ditentukan,
    atau kondisi sumber belum diketahui. Dapat dipaksa lewat kunci 'data_tidak_terverifikasi'."""
    if data.get("data_tidak_terverifikasi"):
        return True
    return (
        not data.get("valid", False)
        or data.get("status") == STATUS_TIDAK_DITENTUKAN
        or _kunci(data.get("kondisi_sumber_air")) == _kunci("Belum diketahui")
    )

def _wajib_berlaku(rec, data, status):
    aturan = rec["wajib"]
    if aturan == "ya":
        return True
    if aturan == "jika mhm aktif":
        return _mhm_aktif(data)
    if aturan == "jika mode krisis dan fitur tersedia":
        return status == STATUS_KRISIS and FITUR_NAVIGATION_AIR_TERSEDIA
    if aturan == "ya jika data tidak terverifikasi":
        return _data_tidak_terverifikasi(data)
    if aturan == "jika ada dugaan kontaminasi":
        # Kalkulator belum memiliki input kontaminasi; hanya wajib jika pemanggil mengirim kuncinya.
        return bool(data.get("dugaan_kontaminasi", False))
    return False

def susun_katalog(data):
    """Filter katalog berdasarkan status, kondisi sumber, MHM, dan kolom aktif.

    Tindakan masuk jika: aktif; status berlaku; kondisi sumber berlaku; syarat MHM terpenuhi;
    dan punya prioritas untuk status itu (atau wajib). Hasil diurutkan: wajib dahulu, lalu prioritas terkecil.
    """
    if not data.get("valid", False):
        data = {**data, "status": STATUS_TIDAK_DITENTUKAN}
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    if status not in KOLOM_PRIORITAS:
        status = STATUS_TIDAK_DITENTUKAN
        data = {**data, "status": status}

    kondisi = _kunci(data.get("kondisi_sumber_air"))
    semua_kondisi = _kunci("Semua kondisi")
    hasil = []
    for rec in KATALOG_RECORDS:
        if not rec["aktif"] or status not in rec["status"]:
            continue
        if semua_kondisi not in rec["kondisi"] and kondisi not in rec["kondisi"]:
            continue
        if rec["syarat_mhm"] and not _mhm_aktif(data):
            continue
        wajib = _wajib_berlaku(rec, data, status)
        prioritas = rec["prioritas"].get(status)
        # Sel prioritas kosong = bukan kandidat biasa untuk status ini; tindakan wajib tetap disertakan.
        if prioritas is None and not wajib:
            continue
        teks = rec["tindakan"].replace(
            "{jumlah_mhm}", str(data.get("jumlah_anggota_mhm", 0))
        ).replace("{sumber}", _teks(data.get("sumber_air_utama")) or "sumber air utama")
        hasil.append({
            "id": rec["id"],
            "tindakan": teks,
            "konteks": rec["konteks"],
            "kategori": rec["kategori"],
            "prioritas": prioritas,
            "wajib": wajib,
            "cta": rec["cta"],
            "catatan_keamanan": rec["catatan_keamanan"],
            "dasar_rujukan": rec["dasar_rujukan"],
        })
    return sorted(
        hasil,
        key=lambda x: (not x["wajib"], x["prioritas"] if x["prioritas"] is not None else math.inf, x["id"]),
    )

def tindakan_wajib(data):
    """Tindakan wajib diputuskan oleh Python, tidak diserahkan kepada Gemini."""
    return [item["id"] for item in susun_katalog(data) if item["wajib"]]

RINGKASAN_PER_STATUS = {
    STATUS_AMAN: "Cadangan diperkirakan mencukupi berdasarkan input dan asumsi prototype. Tetap pantau kondisi dan rencanakan pengisian ulang.",
    STATUS_WASPADA: "Cadangan atau kondisi sumber menunjukkan perlunya pemantauan, penghematan yang aman, dan persiapan alternatif.",
    STATUS_KRISIS: "Sistem mendeteksi kondisi yang membutuhkan tindakan segera. Konfirmasi sumber alternatif sebelum mengandalkannya.",
    STATUS_TIDAK_DITENTUKAN: "Status belum dapat ditentukan karena data belum cukup/valid atau kondisi sumber belum diketahui.",
}
AKSI_UTAMA_PER_STATUS = {
    STATUS_AMAN: "Tanya AIRA Assistant",
    STATUS_WASPADA: "Tanya AIRA Assistant",
    STATUS_KRISIS: "Cari Titik Air melalui Navigation Air",
    STATUS_TIDAK_DITENTUKAN: "Periksa Kembali Data",
}

def pilih_rekomendasi(data):
    """Pilih kandidat rule-based dari katalog: tindakan wajib dahulu, lalu prioritas terkecil."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    if not data.get("valid", False) or status not in KOLOM_PRIORITAS:
        status = STATUS_TIDAK_DITENTUKAN
        data = {**data, "status": status}
    kandidat = susun_katalog(data)
    wajib = [item for item in kandidat if item["wajib"]]
    biasa = [item for item in kandidat if not item["wajib"]]
    dipilih = (wajib + biasa)[:BATAS_REKOMENDASI_RULE]
    # Semua tindakan wajib tetap masuk walau jumlahnya melebihi batas.
    for item in wajib:
        if item not in dipilih:
            dipilih.append(item)
    return {
        "status_sumber": status,
        "ringkasan_rule_based": RINGKASAN_PER_STATUS[status],
        "rekomendasi_id": [item["id"] for item in dipilih],
        "rekomendasi": [item["tindakan"] for item in dipilih],
        "rincian": [
            {k: item[k] for k in ("id", "kategori", "tindakan", "cta", "wajib", "catatan_keamanan", "dasar_rujukan")}
            for item in dipilih
        ],
        "aksi_utama": AKSI_UTAMA_PER_STATUS[status],
        "tautan_fitur_lanjutan": {
            "Navigation Air": status == STATUS_KRISIS,
            "AIRA Assistant": status in {STATUS_AMAN, STATUS_WASPADA, STATUS_KRISIS},
        },
    }

def susun_teks_fallback(rekomendasi):
    daftar = "\n".join(
        f"{i}. {item}" for i, item in enumerate(rekomendasi["rekomendasi"], 1)
    )
    return rekomendasi["ringkasan_rule_based"] + "\n\nLangkah yang bisa dilakukan:\n" + daftar

rekomendasi_rule_based = pilih_rekomendasi(hasil_kalkulasi)
print("Sumber katalog:", SUMBER_KATALOG_AKTIF, f"({len(KATALOG_RECORDS)} tindakan)")
print("Kandidat yang lolos filter:", len(susun_katalog(hasil_kalkulasi)))
print("Tindakan wajib:", tindakan_wajib(hasil_kalkulasi))
print("STATUS:", rekomendasi_rule_based["status_sumber"])
print("AKSI UTAMA:", rekomendasi_rule_based["aksi_utama"])
for i, action in enumerate(rekomendasi_rule_based["rekomendasi"], 1):
    print(f"{i}. {action}")

Sumber katalog: Supabase (tabel katalog_tindakan) (77 tindakan)
Kandidat yang lolos filter: 68
Tindakan wajib: ['jangan_samakan_volume_dengan_kualitas', 'kebersihan_mhm']
STATUS: Waspada
AKSI UTAMA: Tanya AIRA Assistant
1. Jumlah cadangan yang cukup tidak berarti air aman diminum. Periksa informasi kualitas secara terpisah dan ikuti panduan resmi.
2. Pastikan anggota keluarga yang membutuhkan MHM memiliki akses pada air dan perlengkapan kebersihan yang diperlukan serta privasi yang layak.
3. Kurangi penggunaan air untuk kegiatan yang dapat ditunda, tanpa mengabaikan minum, memasak, kebersihan diri, dan sanitasi esensial.
4. Periksa dan catat perkiraan sisa cadangan secara berkala. Perbarui kalkulator jika volume atau penggunaan berubah.
5. Cari pilihan sumber air alternatif dan konfirmasi ketersediaan, akses publik, waktu layanan, biaya jika ada, dan ketentuan penggunaannya.


## 🧪 Validasi Katalog Tindakan dari Supabase

Sel ini memeriksa bahwa pipeline benar-benar memakai katalog dari Supabase dan bahwa aturan filternya bekerja pada skenario yang mewakili tiap status:

- struktur katalog (ID unik, teks terisi, status dan kondisi dikenal) dan ID tindakan yang dirujuk aturan sistem;
- tiap status menghasilkan kandidat;
- tindakan khusus MHM hanya muncul saat MHM aktif, dan tindakan wajib MHM selalu masuk saat MHM aktif;
- Mode Krisis selalu memuat Navigation Air, dan Belum Dapat Ditentukan selalu memuat tindakan memeriksa data;
- tindakan khusus kondisi sumber hanya muncul pada kondisi yang sesuai.

Pemeriksaan ini adalah bukti bahwa kode mengikuti aturan katalog, bukan bukti bahwa isi rekomendasi sudah ditinjau pakar.

In [8]:
# Uji struktur katalog dan aturan filter sebelum rekomendasi dipakai.
ids_katalog = {r["id"] for r in KATALOG_RECORDS}
assert len(KATALOG_RECORDS) > 0, "Katalog kosong."
assert len(ids_katalog) == len(KATALOG_RECORDS), "action_id duplikat."
assert all(r["tindakan"] for r in KATALOG_RECORDS), "Ada teks tindakan kosong."

# ID yang dirujuk aturan sistem harus ada di katalog.
for id_rujukan in (ID_NAVIGATION_AIR, ID_PERIKSA_DATA, ID_MHM_UTAMA, ID_KONFIRMASI_KONDISI, ID_SUMBER_ALTERNATIF):
    assert id_rujukan in ids_katalog, f"ID '{id_rujukan}' dirujuk kode tetapi tidak ada di katalog."

ids_syarat_mhm = {r["id"] for r in KATALOG_RECORDS if r["syarat_mhm"]}
assert ids_syarat_mhm, "Tidak ada tindakan bersyarat MHM di katalog."

# Skenario mewakili tiap status dan kondisi.
s_aman = hitung_ketahanan_air(4, "Sumur", "Normal", volume_cadangan_liter=600)
s_aman_mhm = hitung_ketahanan_air(4, "Sumur", "Normal", volume_cadangan_liter=600, kebutuhan_mhm=True, jumlah_anggota_mhm=2)
s_waspada_normal = hitung_ketahanan_air(2, "Sumur", "Normal", volume_cadangan_liter=60)          # 2 hari, sumber normal
s_waspada_terbatas_mhm = hitung_ketahanan_air(4, "Sumur", "Terbatas", volume_cadangan_liter=450, kebutuhan_mhm=True, jumlah_anggota_mhm=2)
s_krisis = hitung_ketahanan_air(2, "Sumur", "Normal", volume_cadangan_liter=10)
s_krisis_mhm = hitung_ketahanan_air(2, "Sumur", "Tidak berfungsi", volume_cadangan_liter=10, kebutuhan_mhm=True, jumlah_anggota_mhm=1)
s_belum_invalid = hitung_ketahanan_air(0, "Sumur", "Normal", volume_cadangan_liter=100)
s_belum_kondisi = hitung_ketahanan_air(4, "Sumur", "Belum diketahui", volume_cadangan_liter=600)

assert s_aman["status"] == STATUS_AMAN and s_waspada_normal["status"] == STATUS_WASPADA
assert s_krisis["status"] == STATUS_KRISIS and s_krisis_mhm["status"] == STATUS_KRISIS
assert s_belum_invalid["status"] == STATUS_TIDAK_DITENTUKAN and s_belum_kondisi["status"] == STATUS_TIDAK_DITENTUKAN

def ids_k(data):
    return [x["id"] for x in susun_katalog(data)]

def ids_pilihan(data):
    return pilih_rekomendasi(data)["rekomendasi_id"]

pemeriksaan = []

def periksa(nama, kondisi_lulus, detail=""):
    pemeriksaan.append({"Pemeriksaan": nama, "Lulus": bool(kondisi_lulus), "Detail": detail})

for nama, data in [("Aman", s_aman), ("Aman + MHM", s_aman_mhm), ("Waspada", s_waspada_normal),
                   ("Waspada terbatas + MHM", s_waspada_terbatas_mhm), ("Mode Krisis", s_krisis),
                   ("Krisis + MHM", s_krisis_mhm), ("Belum (data tidak valid)", s_belum_invalid),
                   ("Belum (sumber belum diketahui)", s_belum_kondisi)]:
    kandidat = ids_k(data)
    pilihan = ids_pilihan(data)
    periksa(f"{nama}: ada kandidat dan rekomendasi (≥3)", len(kandidat) >= 3 and len(pilihan) >= 3, f"{len(kandidat)} kandidat, {len(pilihan)} dipilih")
    periksa(f"{nama}: semua rekomendasi ada di katalog dan tanpa duplikat",
            set(pilihan) <= ids_katalog and len(pilihan) == len(set(pilihan)))
    periksa(f"{nama}: semua tindakan wajib ikut terpilih", set(tindakan_wajib(data)) <= set(pilihan), str(tindakan_wajib(data)))

# Aturan MHM
periksa("MHM nonaktif: tidak ada tindakan bersyarat MHM", not (set(ids_k(s_aman)) | set(ids_k(s_krisis))) & ids_syarat_mhm)
periksa("MHM aktif: tindakan MHM utama menjadi wajib", ID_MHM_UTAMA in tindakan_wajib(s_aman_mhm) and ID_MHM_UTAMA in tindakan_wajib(s_waspada_terbatas_mhm))
periksa("MHM aktif: tindakan MHM muncul di rekomendasi", ID_MHM_UTAMA in ids_pilihan(s_waspada_terbatas_mhm))

# Aturan status
periksa("Mode Krisis: Navigation Air wajib dan terpilih", ID_NAVIGATION_AIR in tindakan_wajib(s_krisis) and ID_NAVIGATION_AIR in ids_pilihan(s_krisis))
periksa("Belum Dapat Ditentukan: periksa data wajib dan terpilih",
        ID_PERIKSA_DATA in tindakan_wajib(s_belum_invalid) and ID_PERIKSA_DATA in ids_pilihan(s_belum_invalid))
periksa("Status lain tidak memuat tindakan periksa data", ID_PERIKSA_DATA not in ids_k(s_aman) and ID_PERIKSA_DATA not in ids_k(s_krisis))
periksa("Navigation Air tidak wajib saat bukan Mode Krisis", ID_NAVIGATION_AIR not in tindakan_wajib(s_waspada_normal))

# Aturan kondisi sumber
periksa("Sumber normal: tindakan khusus sumber terbatas tidak muncul", ID_SUMBER_ALTERNATIF not in ids_k(s_waspada_normal))
periksa("Sumber terbatas/tidak berfungsi: cari sumber alternatif muncul",
        ID_SUMBER_ALTERNATIF in ids_k(s_waspada_terbatas_mhm) and ID_SUMBER_ALTERNATIF in ids_k(s_krisis_mhm))
periksa("Sumber belum diketahui: konfirmasi kondisi sumber muncul", ID_KONFIRMASI_KONDISI in ids_k(s_belum_kondisi))
periksa("Sumber normal: konfirmasi kondisi sumber tidak muncul", ID_KONFIRMASI_KONDISI not in ids_k(s_waspada_normal))

hasil_validasi_katalog = pd.DataFrame(pemeriksaan)
display(hasil_validasi_katalog[["Pemeriksaan", "Lulus", "Detail"]])
assert hasil_validasi_katalog["Lulus"].all(), "Ada pemeriksaan katalog yang gagal; lihat tabel di atas."
print(f"Semua {len(hasil_validasi_katalog)} pemeriksaan katalog lulus (sumber: {SUMBER_KATALOG_AKTIF}).")

display(pd.DataFrame(susun_katalog(s_waspada_terbatas_mhm))[["id", "kategori", "prioritas", "wajib", "tindakan"]].head(10))

,Pemeriksaan,Lulus,Detail
0,Aman: ada kandidat dan rekomendasi (≥3),True,"39 kandidat, 5 dipilih"
1,Aman: semua rekomendasi ada di katalog dan tan...,True,
2,Aman: semua tindakan wajib ikut terpilih,True,['jangan_samakan_volume_dengan_kualitas']
3,Aman + MHM: ada kandidat dan rekomendasi (≥3),True,"42 kandidat, 5 dipilih"
4,Aman + MHM: semua rekomendasi ada di katalog d...,True,
5,Aman + MHM: semua tindakan wajib ikut terpilih,True,"['jangan_samakan_volume_dengan_kualitas', 'keb..."
6,Waspada: ada kandidat dan rekomendasi (≥3),True,"62 kandidat, 5 dipilih"
7,Waspada: semua rekomendasi ada di katalog dan ...,True,
8,Waspada: semua tindakan wajib ikut terpilih,True,['jangan_samakan_volume_dengan_kualitas']
9,Waspada terbatas + MHM: ada kandidat dan rekom...,True,"68 kandidat, 5 dipilih"


Semua 35 pemeriksaan katalog lulus (sumber: Supabase (tabel katalog_tindakan)).


,id,kategori,prioritas,wajib,tindakan
0,jangan_samakan_volume_dengan_kualitas,Keamanan informasi,1.0,True,Jumlah cadangan yang cukup tidak berarti air a...
1,kebersihan_mhm,MHM,3.0,True,Pastikan anggota keluarga yang membutuhkan MHM...
2,kurangi_penggunaan_non_esensial,Penghematan,1.0,False,Kurangi penggunaan air untuk kegiatan yang dap...
3,pantau_cadangan,Monitoring,1.0,False,Periksa dan catat perkiraan sisa cadangan seca...
4,cari_sumber_alternatif,Akses sumber,2.0,False,Cari pilihan sumber air alternatif dan konfirm...
5,jangan_minum_air_dicurigai,Keamanan air,2.0,False,"Jika air diduga tercemar bahan kimia, limbah, ..."
6,kelompokkan_kebutuhan,Perencanaan,2.0,False,Buat urutan penggunaan air rumah tangga dengan...
7,rencana_isi_ulang,Perencanaan,2.0,False,Rencanakan pengisian ulang sebelum cadangan me...
8,buka_navigation_air,Akses sumber,3.0,False,Gunakan Navigation Air untuk meninjau kandidat...
9,cek_kebocoran,Penghematan,3.0,False,"Periksa keran, sambungan, dan wadah yang terli..."


## 🤖 6. AI Recommendation Layer (Retry, Model Cadangan, dan Fallback)

Pada tahap ini AI (Gemini) **memilih dan mengurutkan 3–5 tindakan** dari katalog tindakan yang sama dengan yang dipakai rule-based (Bagian 5), lalu memberi alasan singkat untuk tiap pilihan. Python tetap menentukan angka dan status; teks tindakan selalu diambil dari katalog sehingga AI tidak dapat menambahkan tindakan baru. Hasil AI divalidasi sebelum ditampilkan: ID di luar katalog dibuang, dan tindakan wajib (misalnya MHM, serta Navigation Air pada Mode Krisis) ditambahkan jika AI melewatkannya.

Data yang dikirim ke AI mencakup kebutuhan dasar, jumlah anggota yang memerlukan MHM, tambahan kebutuhan MHM, total kebutuhan harian, cadangan, estimasi durasi, kondisi sumber, dan status. Lokasi tidak dikirim.

**Kendala yang ditemukan (error 503):** pada uji sebelumnya AI gagal dengan `503 UNAVAILABLE` ("This model is currently experiencing high demand"), padahal uji koneksi berhasil. Ini error sementara di sisi server Gemini, bukan kesalahan kode atau API key, tetapi dulu langsung membuat pipeline jatuh ke fallback. Perbaikannya: semua panggilan AI melalui satu fungsi `panggil_gemini()` yang (1) mencoba ulang error sementara (429/500/502/503/504) dengan jeda bertambah, (2) berpindah ke model cadangan, dan (3) baru memakai rule-based fallback jika semuanya gagal.

Simpan API key di environment variable `GEMINI_API_KEY` atau Colab Secrets. Jangan commit API key. Urutan model dapat diatur lewat `GEMINI_MODELS` (dipisahkan koma); default: `gemini-3.8-flash`, `gemini-3.6-flash`, `gemini-3.5-flash-lite`. Nama model dapat berubah atau dihentikan, jadi cek daftar terbaru di dokumentasi Gemini API. Kegagalan AI tidak boleh menghentikan kalkulator.

Agar prompt tidak terlalu panjang, AI menerima semua tindakan wajib ditambah hingga 25 kandidat berprioritas tertinggi (`BATAS_KANDIDAT_AI`). Validasi akhir tetap memeriksa pilihan AI terhadap seluruh katalog yang lolos filter.

In [9]:
# ===== Konfigurasi Gemini =====
# Urutan = prioritas: model utama dulu, lalu model cadangan jika model utama
# sedang sibuk atau tidak tersedia. Ubah lewat environment variable
# GEMINI_MODELS (dipisahkan koma) tanpa mengedit kode. Nama model bisa

GEMINI_MODELS = [
    nama.strip()
    for nama in os.getenv(
        "GEMINI_MODELS", "gemini-3.8-flash,gemini-3.6-flash,gemini-3.5-flash-lite"
    ).split(",")
    if nama.strip()
]

# Error sementara di sisi server/kuota: layak dicoba ulang.
KODE_ERROR_SEMENTARA = {429, 500, 502, 503, 504}
# Error kredensial: mencoba ulang atau ganti model tidak akan membantu.
KODE_ERROR_KUNCI = {401, 403}


class GeminiTidakTersedia(RuntimeError):
    """AI tidak dapat dipakai (API key/SDK tidak ada, atau semua model gagal)."""


def ambil_api_key():
    """Ambil API key dari environment variable, lalu dari Colab Secrets."""
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key and userdata is not None:
        try:
            api_key = userdata.get("GEMINI_API_KEY")
        except Exception:  # secret belum dibuat / akses notebook belum diizinkan
            api_key = None
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
    return api_key


_cache_client = {}


def get_client():
    """Buat client Gemini satu kali dan pakai ulang."""
    if genai is None:
        raise GeminiTidakTersedia("SDK google-genai belum terpasang")
    api_key = ambil_api_key()
    if not api_key:
        raise GeminiTidakTersedia("API key GEMINI_API_KEY belum tersedia")
    if api_key not in _cache_client:
        _cache_client[api_key] = genai.Client(api_key=api_key)
    return _cache_client[api_key]


def panggil_gemini(
    prompt,
    *,
    json_mode=True,
    temperature=0.2,
    models=None,
    max_percobaan=4,
    jeda_awal=2.0,
    client=None,
    verbose=True,
):
    """Satu-satunya pintu panggilan ke Gemini API.

    Untuk tiap model: error sementara (429/500/502/503/504) dicoba ulang dengan
    jeda yang bertambah (exponential backoff + jitter). Jika masih gagal atau
    modelnya tidak tersedia (mis. 404), lanjut ke model berikutnya.
    Mengembalikan (teks, nama_model); melempar GeminiTidakTersedia jika semua gagal.
    """
    client = client or get_client()
    models = models or GEMINI_MODELS

    config = {"temperature": temperature}
    if json_mode:
        config["response_mime_type"] = "application/json"

    catatan = []
    for model in models:
        for percobaan in range(1, max_percobaan + 1):
            try:
                response = client.models.generate_content(
                    model=model, contents=prompt, config=config
                )
                if not response.text:
                    raise ValueError("Respons AI kosong")
                return response.text, model

            except Exception as exc:
                kode = getattr(exc, "code", None)

                if kode in KODE_ERROR_KUNCI:
                    raise GeminiTidakTersedia(
                        f"API key ditolak (HTTP {kode})"
                    ) from exc

                if kode in KODE_ERROR_SEMENTARA and percobaan < max_percobaan:
                    jeda = jeda_awal * 2 ** (percobaan - 1) * random.uniform(1, 1.25)
                    if verbose:
                        print(
                            f"[Gemini] {model} sibuk (HTTP {kode}), percobaan "
                            f"{percobaan}/{max_percobaan}; ulangi dalam {jeda:.0f} dtk..."
                        )
                    time.sleep(jeda)
                    continue

                catatan.append(f"{model}: {type(exc).__name__} {kode or ''}".strip())
                if verbose:
                    print(f"[Gemini] {model} gagal ({catatan[-1]}); coba model berikutnya.")
                break

    raise GeminiTidakTersedia("Semua model gagal -> " + "; ".join(catatan))


def uji_koneksi_gemini():
    """Cek cepat apakah Gemini dapat dipakai. Tidak menghentikan notebook jika gagal."""
    try:
        teks, model = panggil_gemini(
            "Balas dengan teks: Koneksi Gemini berhasil.",
            json_mode=False,
            max_percobaan=2,
        )
        print(f"Koneksi Gemini berhasil (model: {model}).")
        print("Respons:", teks.strip())
        return True
    except GeminiTidakTersedia as exc:
        print("Gemini belum dapat dipakai:", exc)
        print("Pipeline tetap berjalan dengan rule-based fallback.")
        return False


uji_koneksi_gemini()

[Gemini] gemini-3.8-flash sibuk (HTTP 503), percobaan 1/2; ulangi dalam 2 dtk...
[Gemini] gemini-3.8-flash gagal (gemini-3.8-flash: ServerError 503); coba model berikutnya.
Koneksi Gemini berhasil (model: gemini-3.6-flash).
Respons: Koneksi Gemini berhasil.


True

In [10]:
BATAS_REKOMENDASI_AI = 5
BATAS_KANDIDAT_AI = 25   # jumlah kandidat tidak wajib yang dikirim ke AI (ditambah semua tindakan wajib)

# Data hasil kalkulator yang dikirim ke AI (lokasi sengaja tidak dikirim).
KUNCI_KONTEKS_AI = [
    "status", "alasan_status", "durasi_hari", "cadangan_air_liter",
    "jumlah_anggota", "kondisi_sumber_air", "kebutuhan_mhm",
    "jumlah_anggota_mhm", "kebutuhan_dasar_liter", "tambahan_air_mhm_liter",
    "kebutuhan_harian_liter",
]

def kandidat_untuk_ai(data, batas=BATAS_KANDIDAT_AI):
    """Semua tindakan wajib + kandidat berprioritas tertinggi, dengan kolom ringkas.
    Membatasi daftar mengurangi ukuran prompt dan peluang AI memilih tindakan yang kurang relevan."""
    semua = susun_katalog(data)
    wajib = [x for x in semua if x["wajib"]]
    biasa = [x for x in semua if not x["wajib"]][:batas]
    return [
        {
            "id": x["id"], "kategori": x["kategori"], "tindakan": x["tindakan"],
            "konteks": x["konteks"], "prioritas": x["prioritas"], "wajib": x["wajib"],
            "catatan_keamanan": x["catatan_keamanan"],
        }
        for x in wajib + biasa
    ]


def bangun_prompt_ai(data, rekomendasi, kandidat):
    konteks = {kunci: data.get(kunci) for kunci in KUNCI_KONTEKS_AI}
    id_wajib = [x["id"] for x in kandidat if x["wajib"]]
    return f"""
Anda adalah komponen AI recommendation system untuk AIRA,
sebuah prototipe pendamping ketahanan air rumah tangga.

TUGAS:
Pilih dan urutkan hingga {BATAS_REKOMENDASI_AI} tindakan paling relevan dari kandidat yang
sudah difilter Python. Jangan membuat tindakan baru. Python tetap menjadi
sumber kebenaran untuk perhitungan, status, filter, dan tindakan wajib.

DATA KALKULATOR:
{json.dumps(konteks, ensure_ascii=False)}

RINGKASAN RULE-BASED:
{rekomendasi.get("ringkasan_rule_based", "")}

REKOMENDASI RULE-BASED YANG SUDAH TERSEDIA:
{json.dumps(rekomendasi.get("rekomendasi", []), ensure_ascii=False)}

TINDAKAN WAJIB (ID yang harus tetap ada):
{json.dumps(id_wajib, ensure_ascii=False)}

KATALOG TINDAKAN YANG DIIZINKAN (kolom prioritas: angka lebih kecil = lebih diutamakan):
{json.dumps(kandidat, ensure_ascii=False)}

ATURAN PEMILIHAN:
1. Gunakan status, kondisi sumber air, dan durasi cadangan
   untuk menentukan prioritas tindakan.
2. Pilih hanya ID yang tersedia dalam katalog tindakan di atas.
3. Urutkan ID dari prioritas tertinggi ke terendah.
4. Sertakan semua ID pada daftar TINDAKAN WAJIB; jangan menghapusnya.
   Jika tindakan MHM wajib/berlaku, pastikan tindakan itu tetap disertakan.
5. Jika kondisi sumber air "Belum diketahui", prioritaskan tindakan
   "{ID_KONFIRMASI_KONDISI}" bila tersedia. Jangan menganggap sumber
   air pasti tersedia atau tidak berfungsi.
6. Jika kondisi sumber air "Tidak berfungsi", prioritaskan persiapan
   sumber alternatif (mis. "{ID_SUMBER_ALTERNATIF}") bila tersedia.
7. Jika status "Mode Krisis", sertakan tindakan "{ID_NAVIGATION_AIR}".
8. Jika status "Belum Dapat Ditentukan", sertakan "{ID_PERIKSA_DATA}",
   jelaskan keterbatasan informasi dan jangan memberikan
   kepastian yang tidak didukung data.
9. Jangan mengubah, menghitung ulang, atau menyimpulkan status
   kalkulator yang berbeda dari hasil Python.
10. Jangan membuat angka, lokasi, titik air, jam buka, biaya,
    atau klaim kualitas air yang tidak tersedia dalam data.
11. Jangan menyatakan air aman diminum hanya berdasarkan hasil
    kalkulator ketahanan air.
12. Berikan alasan singkat untuk setiap pilihan yang mengacu
    pada konteks pengguna.
13. Gunakan Bahasa Indonesia yang sederhana, empatik,
    dan tidak menghakimi.

KELUARAN:
Kembalikan JSON valid saja dengan struktur:
{{
  "ringkasan": "penjelasan singkat kondisi pengguna",
  "selected_ids": ["id_prioritas_1", "id_prioritas_2"],
  "alasan": {{
    "id_prioritas_1": "alasan singkat",
    "id_prioritas_2": "alasan singkat"
  }}
}}

Jangan memasukkan ID yang tidak ada dalam katalog.
"""


def parse_json_ai(teks):
    """Parse JSON dari AI; toleran terhadap pembungkus ```json ... ```."""
    teks = teks.strip()
    if teks.startswith("```"):
        teks = teks.strip("`").strip()
        if teks.lower().startswith("json"):
            teks = teks[4:]
    hasil = json.loads(teks)
    if not isinstance(hasil, dict):
        raise ValueError("Format JSON AI tidak valid")
    return hasil


def pilih_id_final(id_dari_ai, katalog_by_id, wajib, batas=BATAS_REKOMENDASI_AI):
    """Validasi pilihan AI: buang ID asing/duplikat, tambahkan tindakan wajib,
    lalu batasi jumlahnya tanpa pernah membuang tindakan wajib."""
    if not isinstance(id_dari_ai, list):
        raise ValueError("Format selected_ids tidak valid")

    pilihan = list(dict.fromkeys(
        item for item in id_dari_ai
        if isinstance(item, str) and item in katalog_by_id
    ))
    if not pilihan:
        raise ValueError("AI tidak memilih rekomendasi yang valid")

    for item in wajib:
        if item not in pilihan:
            pilihan.append(item)

    for item in reversed(list(pilihan)):
        if len(pilihan) <= batas:
            break
        if item not in wajib:
            pilihan.remove(item)

    return pilihan


def generate_ai_recommendation(data, rekomendasi):
    """AI recommendation system untuk memilih dan memprioritaskan tindakan AIRA.

    Python tetap menentukan hasil kalkulasi dan status. Gemini memilih dan
    mengurutkan tindakan dari katalog Supabase yang diizinkan serta memberi alasan.
    Teks tindakan selalu berasal dari katalog (AI tidak dapat membuat tindakan
    baru). Jika AI tidak tersedia atau keluarannya tidak valid, sistem memakai
    rekomendasi rule-based sebagai fallback.
    """
    fallback = susun_teks_fallback(rekomendasi)

    try:
        katalog = susun_katalog(data)
        katalog_by_id = {item["id"]: item for item in katalog}
        kandidat = kandidat_untuk_ai(data)

        teks_json, model = panggil_gemini(bangun_prompt_ai(data, rekomendasi, kandidat))
        hasil_ai = parse_json_ai(teks_json)

        selected_ids = pilih_id_final(
            hasil_ai.get("selected_ids", []), katalog_by_id, tindakan_wajib(data)
        )

        alasan_ai = hasil_ai.get("alasan")
        if not isinstance(alasan_ai, dict):
            alasan_ai = {}

        ringkasan_ai = hasil_ai.get("ringkasan")
        if not isinstance(ringkasan_ai, str) or not ringkasan_ai.strip():
            ringkasan_ai = rekomendasi.get("ringkasan_rule_based", "")

        baris = []
        for nomor, item_id in enumerate(selected_ids, start=1):
            teks_item = f"{nomor}. {katalog_by_id[item_id]['tindakan']}"
            alasan = alasan_ai.get(item_id, "")
            if isinstance(alasan, str) and alasan.strip():
                teks_item += f"\n   Alasan: {alasan.strip()}"
            baris.append(teks_item)

        return {
            "mode": f"AI recommendation ({model})",
            "teks": (
                ringkasan_ai.strip()
                + "\n\nRekomendasi prioritas AIRA:\n"
                + "\n".join(baris)
            ),
            "rekomendasi_ai": selected_ids,
            "rincian": [
                {k: katalog_by_id[i][k] for k in ("id", "kategori", "cta", "wajib", "catatan_keamanan", "dasar_rujukan")}
                for i in selected_ids
            ],
        }

    except Exception as exc:
        return {
            "mode": f"rule-based fallback (AI gagal: {type(exc).__name__}: {exc})",
            "teks": fallback,
            "rekomendasi_ai": [],
            "rincian": rekomendasi.get("rincian", []),
        }


# Jalankan AI recommendation system
hasil_penyampaian = generate_ai_recommendation(hasil_kalkulasi, rekomendasi_rule_based)

print("MODE:", hasil_penyampaian["mode"])
print("\n" + hasil_penyampaian["teks"])

[Gemini] gemini-3.8-flash sibuk (HTTP 503), percobaan 1/4; ulangi dalam 2 dtk...
MODE: AI recommendation (gemini-3.8-flash)

Cadangan air diperkirakan cukup untuk sekitar 6 hari, namun status Waspada berlaku karena sumber air utama terbatas dan ada kebutuhan MHM dalam keluarga.

Rekomendasi prioritas AIRA:
1. Jumlah cadangan yang cukup tidak berarti air aman diminum. Periksa informasi kualitas secara terpisah dan ikuti panduan resmi.
   Alasan: Tindakan wajib untuk mengingatkan bahwa estimasi volume tidak menjamin kelayakan atau keamanan air minum.
2. Kurangi penggunaan air untuk kegiatan yang dapat ditunda, tanpa mengabaikan minum, memasak, kebersihan diri, dan sanitasi esensial.
   Alasan: Membantu menjaga ketahanan cadangan air selama pasokan utama sedang terbatas tanpa mengorbankan kebutuhan pokok.
3. Periksa dan catat perkiraan sisa cadangan secara berkala. Perbarui kalkulator jika volume atau penggunaan berubah.
   Alasan: Diperlukan untuk memantau sisa volume tersimpan secara be

## 🧪 7. Pengujian Pipeline

Pengujian berikut memeriksa batas klasifikasi (kurang dari 1 hari, tepat 1 hari, tepat 3 hari, dan lebih dari 3 hari), kondisi sumber yang tidak berfungsi/terbatas/belum diketahui, input tidak valid, serta perhitungan tambahan kebutuhan MHM. Lapisan AI diuji pada sel terpisah di bawahnya tanpa memanggil API (respons disimulasikan): retry, model cadangan, validasi ID, tindakan wajib, dan fallback.

Pengujian MHM memastikan bahwa kebutuhan dasar dan tambahan MHM dijumlahkan sebelum durasi cadangan dihitung, serta jumlah anggota yang memerlukan MHM tidak melebihi jumlah seluruh anggota keluarga. Ini merupakan bukti awal bahwa fungsi mengikuti aturan yang dinyatakan, bukan bukti bahwa angka kebutuhan maupun ambang status sudah tervalidasi untuk semua kondisi nyata.

In [11]:
def skenario_durasi(durasi, kondisi="Normal"):
    # 1 orang × 15 L/hari; volume cadangan disusun agar durasinya sesuai target.
    return hitung_ketahanan_air(
        jumlah_anggota=1,
        sumber_air_utama="Simulasi",
        kondisi_sumber_air=kondisi,
        volume_cadangan_liter=durasi * KEBUTUHAN_PER_ORANG_LITER_HARI,
    )


# Skenario perhitungan MHM:
# 4 anggota, 2 memerlukan MHM, cadangan 450 liter.
hasil_uji_mhm = hitung_ketahanan_air(
    jumlah_anggota=4,
    sumber_air_utama="Sumur",
    kondisi_sumber_air="Normal",
    volume_cadangan_liter=450,
    kebutuhan_mhm=True,
    jumlah_anggota_mhm=2,
)

# Skenario input tidak valid: jumlah anggota MHM melebihi jumlah keluarga.
hasil_uji_jumlah_mhm_tidak_valid = hitung_ketahanan_air(
    jumlah_anggota=2,
    sumber_air_utama="Sumur",
    kondisi_sumber_air="Normal",
    volume_cadangan_liter=100,
    kebutuhan_mhm=True,
    jumlah_anggota_mhm=3,
)


test_cases = [
    ("0.5 hari → Krisis", skenario_durasi(0.5), STATUS_KRISIS),
    ("1 hari → Waspada", skenario_durasi(1), STATUS_WASPADA),
    ("3 hari → Waspada", skenario_durasi(3), STATUS_WASPADA),
    ("3.01 hari → Aman", skenario_durasi(3.01), STATUS_AMAN),
    (
        "Sumber tidak berfungsi → Krisis",
        skenario_durasi(5, "Tidak berfungsi"),
        STATUS_KRISIS,
    ),
    (
        "Sumber terbatas membatasi status aman",
        skenario_durasi(5, "Terbatas"),
        STATUS_WASPADA,
    ),
    (
        "Kondisi sumber belum diketahui",
        skenario_durasi(5, "Belum diketahui"),
        STATUS_TIDAK_DITENTUKAN,
    ),
    (
        "Anggota nol → tidak ditentukan",
        hitung_ketahanan_air(
            0, "Sumur", "Normal", volume_cadangan_liter=100
        ),
        STATUS_TIDAK_DITENTUKAN,
    ),
    (
        "Persentase >100 → tidak ditentukan",
        hitung_ketahanan_air(
            2, "Sumur", "Normal",
            kapasitas_tangki_liter=500,
            persentase_isi=120,
        ),
        STATUS_TIDAK_DITENTUKAN,
    ),
    (
        "MHM dihitung dalam kebutuhan harian",
        hasil_uji_mhm,
        STATUS_AMAN,
    ),
    (
        "Jumlah anggota MHM melebihi jumlah keluarga",
        hasil_uji_jumlah_mhm_tidak_valid,
        STATUS_TIDAK_DITENTUKAN,
    ),
]

rows = []

for nama, hasil, expected in test_cases:
    actual = hasil["status"]
    rows.append({
        "Skenario": nama,
        "Status aktual": actual,
        "Status yang diharapkan": expected,
        "Lulus": actual == expected,
    })

test_results = pd.DataFrame(rows)
display(test_results)

assert test_results["Lulus"].all(), (
    "Ada pengujian yang gagal; periksa aturan klasifikasi/validasi."
)

# Periksa nilai perhitungan MHM secara terpisah.
assert hasil_uji_mhm["valid"] is True
assert abs(hasil_uji_mhm["kebutuhan_dasar_liter"] - 60) < 1e-9
assert abs(hasil_uji_mhm["tambahan_air_mhm_liter"] - 10) < 1e-9
assert abs(hasil_uji_mhm["kebutuhan_harian_liter"] - 70) < 1e-9
assert hasil_uji_mhm["durasi_hari"] == round(450 / 70, 2)

assert hasil_uji_jumlah_mhm_tidak_valid["valid"] is False

print(
    f"Semua {len(test_results)} pengujian status lulus, "
    "dan pengujian nilai perhitungan MHM lulus."
)

,Skenario,Status aktual,Status yang diharapkan,Lulus
0,0.5 hari → Krisis,Mode Krisis,Mode Krisis,True
1,1 hari → Waspada,Waspada,Waspada,True
2,3 hari → Waspada,Waspada,Waspada,True
3,3.01 hari → Aman,Aman / Relatif Mencukupi,Aman / Relatif Mencukupi,True
4,Sumber tidak berfungsi → Krisis,Mode Krisis,Mode Krisis,True
5,Sumber terbatas membatasi status aman,Waspada,Waspada,True
6,Kondisi sumber belum diketahui,Belum Dapat Ditentukan,Belum Dapat Ditentukan,True
7,Anggota nol → tidak ditentukan,Belum Dapat Ditentukan,Belum Dapat Ditentukan,True
8,Persentase >100 → tidak ditentukan,Belum Dapat Ditentukan,Belum Dapat Ditentukan,True
9,MHM dihitung dalam kebutuhan harian,Aman / Relatif Mencukupi,Aman / Relatif Mencukupi,True


Semua 11 pengujian status lulus, dan pengujian nilai perhitungan MHM lulus.


In [12]:
# --- Pengujian lapisan AI TANPA memanggil API (respons disimulasikan) ---

class _ErrorAPI(Exception):
    """Meniru error Gemini API yang membawa kode HTTP."""
    def __init__(self, code):
        super().__init__(f"HTTP {code}")
        self.code = code


class _ClientPalsu:
    """Client palsu: tiap panggilan mengembalikan/melempar item berikutnya."""
    def __init__(self, urutan):
        self.urutan = list(urutan)
        self.dipanggil = []
        self.models = self

    def generate_content(self, model, contents, config=None):
        self.dipanggil.append(model)
        aksi = self.urutan.pop(0)
        if isinstance(aksi, Exception):
            raise aksi
        return type("Respons", (), {"text": aksi})()


def _uji_panggil(urutan, **opsi):
    client = _ClientPalsu(urutan)
    try:
        hasil = panggil_gemini(
            "x", models=["utama", "cadangan"], client=client,
            jeda_awal=0, verbose=False, **opsi,
        )
    except GeminiTidakTersedia as exc:
        hasil = exc
    return hasil, client.dipanggil


def _uji_ai(data, respons):
    """Jalankan generate_ai_recommendation dengan respons Gemini palsu."""
    asli = globals()["panggil_gemini"]

    def palsu(prompt, **kwargs):
        if isinstance(respons, Exception):
            raise respons
        return respons, "model-simulasi"

    globals()["panggil_gemini"] = palsu
    try:
        return generate_ai_recommendation(data, pilih_rekomendasi(data))
    finally:
        globals()["panggil_gemini"] = asli


data_waspada_mhm = hitung_ketahanan_air(
    4, "Sumur", "Terbatas", volume_cadangan_liter=450,
    kebutuhan_mhm=True, jumlah_anggota_mhm=2,
)
data_krisis = skenario_durasi(0.5)
data_belum = hitung_ketahanan_air(0, "Sumur", "Normal", volume_cadangan_liter=100)

def ids_k(data):
    return [x["id"] for x in susun_katalog(data)]

# ID dalam respons simulasi diambil dari katalog nyata agar tes tidak bergantung pada nama ID tertentu.
wajib_waspada = tindakan_wajib(data_waspada_mhm)
biasa_waspada = [i for i in ids_k(data_waspada_mhm) if i not in wajib_waspada]
wajib_krisis = tindakan_wajib(data_krisis)
wajib_belum = tindakan_wajib(data_belum)
id_bukan_krisis = ID_PERIKSA_DATA          # hanya berlaku untuk Belum Dapat Ditentukan
assert id_bukan_krisis not in ids_k(data_krisis)
assert len(biasa_waspada) >= 6, "Katalog terlalu sedikit untuk menguji batas 5 tindakan."

# Retry & model cadangan
h1, d1 = _uji_panggil([_ErrorAPI(503), _ErrorAPI(503), "{}"])
h2, d2 = _uji_panggil([_ErrorAPI(503), _ErrorAPI(503), "{}"], max_percobaan=2)
h3, d3 = _uji_panggil([_ErrorAPI(404), "{}"])
h4, d4 = _uji_panggil([_ErrorAPI(403)])
h5, d5 = _uji_panggil([_ErrorAPI(503)] * 2, max_percobaan=1)

# Guardrail hasil AI
ai_a = _uji_ai(data_waspada_mhm, json.dumps({
    "ringkasan": "Ringkasan AI.",
    "selected_ids": ["id_palsu"] + biasa_waspada[:6],      # enam tindakan biasa + satu ID asing, tanpa tindakan wajib
    "alasan": {biasa_waspada[0]: "Sumber sedang terbatas."},
}))
ai_b = _uji_ai(data_krisis, json.dumps({
    "ringkasan": "Ringkasan AI.",
    "selected_ids": [id_bukan_krisis, wajib_krisis[0]],
}))
ai_e = _uji_ai(data_belum, json.dumps({
    "ringkasan": "Ringkasan AI.",
    "selected_ids": [ids_k(data_belum)[-1]],
}))
ai_c = _uji_ai(data_krisis, _ErrorAPI(503))
ai_d = _uji_ai(data_krisis, "ini bukan JSON")

uji_ai = [
    ("503 dicoba ulang pada model yang sama", h1 == ("{}", "utama") and d1 == ["utama"] * 3),
    ("503 berulang → pindah ke model cadangan", h2 == ("{}", "cadangan") and d2 == ["utama", "utama", "cadangan"]),
    ("404 (model tidak ada) → langsung model cadangan", h3 == ("{}", "cadangan") and d3 == ["utama", "cadangan"]),
    ("403 (API key ditolak) → berhenti tanpa retry", isinstance(h4, GeminiTidakTersedia) and d4 == ["utama"]),
    ("Semua model gagal → GeminiTidakTersedia", isinstance(h5, GeminiTidakTersedia)),
    ("ID di luar katalog dibuang", "id_palsu" not in ai_a["rekomendasi_ai"]),
    ("Maksimal 5 tindakan bila jumlah wajib tidak melebihi batas",
     len(ai_a["rekomendasi_ai"]) == max(BATAS_REKOMENDASI_AI, len(wajib_waspada))),
    ("Tindakan wajib (termasuk MHM) tetap ada walau AI melewatkannya",
     set(wajib_waspada) <= set(ai_a["rekomendasi_ai"]) and ID_MHM_UTAMA in ai_a["rekomendasi_ai"]),
    ("Mode Krisis: Navigation Air ditambahkan dan semua tindakan wajib masuk",
     ID_NAVIGATION_AIR in ai_b["rekomendasi_ai"] and set(wajib_krisis) <= set(ai_b["rekomendasi_ai"])),
    ("Mode Krisis: tindakan khusus 'Belum Dapat Ditentukan' ditolak", id_bukan_krisis not in ai_b["rekomendasi_ai"]),
    ("Belum Dapat Ditentukan: periksa data ditambahkan walau AI melewatkannya",
     ID_PERIKSA_DATA in ai_e["rekomendasi_ai"] and set(wajib_belum) <= set(ai_e["rekomendasi_ai"])),
    ("Hasil AI memuat rincian katalog (CTA dan dasar rujukan)",
     all("cta" in r and "dasar_rujukan" in r for r in ai_a["rincian"]) and len(ai_a["rincian"]) == len(ai_a["rekomendasi_ai"])),
    ("Respons AI berupa error → rule-based fallback",
     ai_c["mode"].startswith("rule-based fallback") and ai_c["rekomendasi_ai"] == []),
    ("Respons AI bukan JSON → rule-based fallback", ai_d["mode"].startswith("rule-based fallback")),
    ("Prompt AI hanya memuat kandidat terbatas dan menyebut ID wajib",
     len(kandidat_untuk_ai(data_waspada_mhm)) <= BATAS_KANDIDAT_AI + len(wajib_waspada)
     and all(w in bangun_prompt_ai(data_waspada_mhm, pilih_rekomendasi(data_waspada_mhm), kandidat_untuk_ai(data_waspada_mhm)) for w in wajib_waspada)),
]

hasil_uji_ai = pd.DataFrame(uji_ai, columns=["Skenario lapisan AI", "Lulus"])
display(hasil_uji_ai)
assert hasil_uji_ai["Lulus"].all(), "Ada pengujian lapisan AI yang gagal."
print(f"Semua {len(hasil_uji_ai)} pengujian lapisan AI lulus (tanpa memanggil API).")

,Skenario lapisan AI,Lulus
0,503 dicoba ulang pada model yang sama,True
1,503 berulang → pindah ke model cadangan,True
2,404 (model tidak ada) → langsung model cadangan,True
3,403 (API key ditolak) → berhenti tanpa retry,True
4,Semua model gagal → GeminiTidakTersedia,True
5,ID di luar katalog dibuang,True
6,Maksimal 5 tindakan bila jumlah wajib tidak me...,True
7,Tindakan wajib (termasuk MHM) tetap ada walau ...,True
8,Mode Krisis: Navigation Air ditambahkan dan se...,True
9,Mode Krisis: tindakan khusus 'Belum Dapat Dite...,True


Semua 15 pengujian lapisan AI lulus (tanpa memanggil API).


## 📊 8. Interpretasi Hasil Simulasi

Baca hasil skenario utama di atas dengan membedakan tiga hal:

1. **Angka hasil kalkulasi** dihitung oleh fungsi Python. Kebutuhan air dasar menggunakan acuan 15 liter/orang/hari untuk konteks darurat. Jika ada anggota keluarga yang memerlukan MHM, tambahan 5 liter/orang/hari per anggota tersebut ikut dihitung sesuai alokasi operasional AIRA.
2. **Status** dipilih oleh aturan sistem berdasarkan durasi dan kondisi sumber air. Contohnya, cadangan dapat diperkirakan cukup lebih dari 3 hari tetapi status menjadi Waspada bila sumber utama sedang terbatas.
3. **Rekomendasi** dipilih dari satu katalog tindakan. Rule-based memilih secara deterministik; AI, jika aktif, hanya memilih dan mengurutkan tindakan dari katalog yang sama beserta alasannya; jika AI gagal, teks rule-based tetap tersedia.

Hasil adalah estimasi berdasarkan input, bukan pengukuran sensor real-time. Kondisi sumber air dan cadangan dapat berubah. Kalkulator tidak menentukan kualitas atau keamanan air minum. Angka tambahan MHM merupakan alokasi operasional AIRA dan tidak boleh dianggap sebagai standar minimum universal yang telah ditetapkan secara khusus oleh Sphere atau UNICEF.

In [13]:
# Ringkasan terstruktur yang mudah disimpan atau ditampilkan sebagai output awal.
ringkasan_output = {
    "input": {
        "lokasi": hasil_kalkulasi.get("lokasi"),
        "jumlah_anggota": hasil_kalkulasi.get("jumlah_anggota"),
        "sumber_air_utama": hasil_kalkulasi.get("sumber_air_utama"),
        "kondisi_sumber_air": hasil_kalkulasi.get("kondisi_sumber_air"),
        "metode_volume": hasil_kalkulasi.get("metode_volume"),
        "kebutuhan_mhm": hasil_kalkulasi.get("kebutuhan_mhm"),
        "jumlah_anggota_mhm": hasil_kalkulasi.get("jumlah_anggota_mhm"),
    },
    "hasil_kalkulasi": {
        "kebutuhan_dasar_liter": hasil_kalkulasi.get("kebutuhan_dasar_liter"),
        "tambahan_air_mhm_liter": hasil_kalkulasi.get("tambahan_air_mhm_liter"),
        "kebutuhan_harian_liter": hasil_kalkulasi.get("kebutuhan_harian_liter"),
        "cadangan_air_liter": hasil_kalkulasi.get("cadangan_air_liter"),
        "durasi_hari": hasil_kalkulasi.get("durasi_hari"),
        "status": hasil_kalkulasi.get("status"),
        "alasan_status": hasil_kalkulasi.get("alasan_status"),
    },
    "sumber_katalog": SUMBER_KATALOG_AKTIF,
    "rekomendasi": rekomendasi_rule_based,
    "waktu_perhitungan_utc": hasil_kalkulasi.get("waktu_perhitungan_utc"),
}

print(json.dumps(ringkasan_output, ensure_ascii=False, indent=2))

{
  "input": {
    "lokasi": "Rongkop, Gunungkidul",
    "jumlah_anggota": 4,
    "sumber_air_utama": "Sumur",
    "kondisi_sumber_air": "Terbatas",
    "metode_volume": "kapasitas × persentase",
    "kebutuhan_mhm": true,
    "jumlah_anggota_mhm": 2
  },
  "hasil_kalkulasi": {
    "kebutuhan_dasar_liter": 60.0,
    "tambahan_air_mhm_liter": 10.0,
    "kebutuhan_harian_liter": 70.0,
    "cadangan_air_liter": 450.0,
    "durasi_hari": 6.43,
    "status": "Waspada",
    "alasan_status": "Cadangan tersimpan diperkirakan cukup lebih dari 3 hari, tetapi sumber air utama sedang terbatas."
  },
  "sumber_katalog": "Supabase (tabel katalog_tindakan)",
  "rekomendasi": {
    "status_sumber": "Waspada",
    "ringkasan_rule_based": "Cadangan atau kondisi sumber menunjukkan perlunya pemantauan, penghematan yang aman, dan persiapan alternatif.",
    "rekomendasi_id": [
      "jangan_samakan_volume_dengan_kualitas",
      "kebersihan_mhm",
      "kurangi_penggunaan_non_esensial",
      "pantau_cadan

## ⚠️ 9. Keterbatasan dan Responsible AI

### Keterbatasan kalkulator dan data
- **Acuan kebutuhan dasar:** Kalkulator menggunakan 15 liter/orang/hari sebagai acuan kebutuhan air minimum dalam konteks darurat. Angka ini bukan kebutuhan ideal untuk semua aktivitas rumah tangga dan tidak selalu mencerminkan kebutuhan aktual setiap wilayah.
- **MHM:** Alokasi 5 liter/orang/hari untuk setiap anggota keluarga yang memerlukan Manajemen Kebersihan Menstruasi (MHM) adalah keputusan operasional AIRA, bukan angka minimum universal yang secara khusus ditetapkan oleh Sphere atau UNICEF. Besaran dan penerapannya perlu dievaluasi berdasarkan panduan yang relevan serta kondisi lokal.
- **Kondisi sumber:** Kategori Normal/Terbatas/Tidak berfungsi/Belum diketahui berasal dari input pengguna, bukan sensor atau API real-time.
- **Ambang status:** Batas 1 dan 3 hari adalah aturan awal prototype, bukan klasifikasi resmi kekeringan.
- **Kualitas air:** Volume dan durasi tidak menunjukkan apakah air aman diminum.
- **Data uji:** Input pada notebook ini adalah skenario simulasi, bukan data rumah tangga nyata. Jangan menyimpan data yang dapat mengidentifikasi pengguna di repository publik tanpa dasar dan izin yang sesuai.
- **Di luar cakupan pipeline ini:** Simulasi panen air hujan (luas atap dan curah hujan) baru ditambahkan setelah metode dan sumber datanya ditetapkan.
- **Isi katalog tindakan:** katalog dikelola tim di Supabase dan belum ditinjau pakar air, sanitasi, atau MHM. Kolom `dasar_rujukan` pada sebagian besar tindakan masih berupa rujukan umum (misalnya "WHO/UNICEF/CDC; adaptasi AIRA") yang belum menunjuk dokumen atau halaman tertentu.
- **Aturan wajib tanpa input:** tindakan wajib "Jika ada dugaan kontaminasi" belum pernah aktif karena kalkulator belum memiliki input dugaan kontaminasi. Aturan "fitur tersedia" untuk Navigation Air dianggap selalu terpenuhi (`FITUR_NAVIGATION_AIR_TERSEDIA`).
- **Sumber katalog saat pengujian:** pengujian berjalan pada sumber yang tercetak di output sel pemuatan katalog. Jika yang tercetak CSV cadangan, isinya dapat berbeda dari tabel Supabase.

### Responsible AI: bagaimana AI dibatasi pada pipeline ini
- **Angka dan status berasal dari kode.** AI tidak menghitung ulang dan tidak boleh mengubah angka atau status.
- **AI hanya memilih dari katalog.** Teks tindakan selalu berasal dari katalog; ID di luar katalog dibuang dan tindakan wajib (misalnya MHM, serta Navigation Air pada Mode Krisis) tetap ditambahkan bila AI melewatkannya.
- **Ada jalur cadangan.** Jika AI gagal atau keluarannya tidak valid, rekomendasi rule-based tetap ditampilkan sebagai referensi utama.
- **Data yang dikirim dibatasi.** Hanya hasil kalkulasi yang dikirim ke AI; lokasi tidak dikirim.
- **Keluaran LLM bisa keliru.** Alasan yang ditulis AI belum diverifikasi otomatis dan perlu ditinjau manusia. Pengujian dengan API asli baru dilakukan pada satu skenario.

## 🚀 10. Next Steps

### Kendala yang ditemukan
- **Model AI sibuk (error 503).** Pada uji awal Gemini menolak permintaan dengan `503 UNAVAILABLE` sehingga pipeline hanya memakai fallback. Perbaikannya: satu fungsi panggilan AI dengan retry bertahap, model cadangan, dan fallback rule-based. Konsekuensinya, waktu respons AI bisa lebih lama saat layanan padat.
- **Nama model dapat berubah.** Nama model dan jadwal penghentiannya ditentukan penyedia, sehingga daftar `GEMINI_MODELS` perlu dipantau dan diperbarui.
- **Pengujian AI masih terbatas.** Output AI dengan API asli baru diuji pada satu skenario, dan kualitas alasan yang ditulis AI belum dinilai secara sistematis.

### Rencana pengembangan berikutnya

**Validasi dan pengujian**
1. Tinjau acuan 15 liter/orang/hari dan ambang status dengan sumber yang relevan serta mentor/tim.
2. Evaluasi alokasi tambahan 5 liter/orang/hari untuk MHM dengan merujuk pada panduan kebersihan menstruasi dan kondisi lokal sebelum digunakan untuk rekomendasi nyata.
3. Uji AI dengan API asli pada skenario Krisis dan Belum Dapat Ditentukan, termasuk skenario tanpa MHM dan dengan beberapa anggota keluarga yang memerlukan MHM.

**Integrasi dengan fitur lain**
4. Hubungkan output kalkulator ke AIRA Assistant dengan mengirim hasil terstruktur (kebutuhan dasar, tambahan MHM, total kebutuhan harian, durasi, status, alasan, rekomendasi). Assistant menjelaskan hasil tanpa menghitung ulang.
5. Integrasikan Navigation Air sebagai aksi lanjutan saat Mode Krisis, dengan catatan titik air harus diverifikasi dan ketersediaannya dikonfirmasi.

**Pengembangan lanjutan**
6. Evaluasi apakah kondisi sumber air perlu menjadi pemicu status atau ditampilkan sebagai indikator terpisah; aturan saat ini adalah keputusan prototype yang harus divalidasi.
7. Lakukan uji pengguna agar label, satuan, dan pesan kesalahan mudah dipahami, serta tambahkan petunjuk memperkirakan volume tangki sesuai desain Figma.

**Pengumpulan**
8. Simpan notebook, README, dan laporan pengujian ke folder repository GitHub kelompok; jangan commit API key atau data pribadi.

**Katalog tindakan**
9. Isi `dasar_rujukan` dengan dokumen dan halaman spesifik (dapat dihubungkan dengan tabel `document_registry` yang dipakai AIRA Assistant), lalu minta tinjauan pakar atas teks tindakan dan catatan keamanannya.
10. Tambahkan input dugaan kontaminasi pada kalkulator atau tentukan perlakuan aturan wajib "Jika ada dugaan kontaminasi", dan hubungkan aturan "fitur tersedia" dengan status Navigation Air yang sebenarnya.

## 🏁 Kesimpulan Pipeline Kalkulator Ketahanan Air AIRA

**Apa yang dibangun.** Pipeline Kalkulator Ketahanan Air mengolah input rumah tangga melalui validasi, perhitungan kebutuhan dan durasi cadangan, klasifikasi status, pemilihan rekomendasi rule-based dari katalog Supabase, lalu pemilihan/penyusunan urutan rekomendasi oleh Gemini. Perhitungan angka dan status tetap dilakukan oleh Python; AI tidak diberi kewenangan mengubah keduanya atau menambahkan tindakan di luar katalog.

### 📊 Hasil skenario utama berdasarkan eksekusi notebook

Skenario simulasi menggunakan rumah tangga 4 orang di Rongkop, Gunungkidul, sumber air sumur dengan kondisi terbatas, cadangan 450 liter (tangki 1.000 liter terisi 45%), dan 2 anggota keluarga yang memerlukan MHM.

- Kebutuhan dasar: **60 liter/hari** (4 × 15 liter).
- Tambahan operasional MHM: **10 liter/hari** (2 × 5 liter).
- Total kebutuhan: **70 liter/hari**.
- Estimasi cadangan: **450 liter**, cukup sekitar **6,43 hari**.
- Status berdasarkan durasi saja: **Aman / Relatif Mencukupi**. Status akhir menjadi **Waspada** karena sumber air utama berstatus **Terbatas**.

### Perbandingan rekomendasi rule-based dan AI

Kedua pendekatan menggunakan sumber katalog yang sama, yaitu tabel Supabase `katalog_tindakan` berisi **77 tindakan aktif**. Setelah filter status, kondisi sumber, dan MHM, terdapat **68 kandidat**.

| Aspek | Rule-based (Bagian 5) | AI (Bagian 6) |
|---|---|---|
| Jumlah rekomendasi | 5 tindakan | 5 tindakan |
| Tindakan yang dipilih | Kualitas air; kebersihan MHM; penghematan non-esensial; pemantauan cadangan; sumber alternatif | Lima tindakan yang sama |
| Urutan awal | 1) Kualitas air, 2) MHM, 3) Penghematan, 4) Pemantauan, 5) Sumber alternatif | 1) Kualitas air, 2) Penghematan, 3) Pemantauan, 4) Sumber alternatif, 5) MHM |
| Kekuatan | Urutan konsisten berdasarkan aturan dan prioritas katalog | Menyesuaikan urutan dengan konteks dan memberi alasan bahasa alami |
| Batasan | Tidak menghasilkan alasan kontekstual seluwes AI | Urutan dapat berbeda; tetap perlu pemeriksaan manusia dan tidak otomatis lebih benar |

**Interpretasi pembandingan:** pada skenario ini AI tidak menambah tindakan baru dan tidak menghilangkan dua tindakan wajib: menjaga pemisahan antara kuantitas dan kualitas air serta menjaga kebutuhan MHM. Perbedaan utamanya adalah urutan. Rule-based menempatkan MHM pada urutan kedua, sedangkan AI menempatkannya pada urutan kelima. Karena MHM merupakan tindakan wajib dan menyangkut kebutuhan kebersihan serta martabat pengguna, penempatan di urutan terakhir perlu ditinjau oleh tim/pakar sebelum rekomendasi digunakan pada produk. AI memberi alasan untuk setiap tindakan, tetapi alasan tersebut tetap perlu ditinjau dan tidak boleh dianggap sebagai validasi pakar.

**Ketersediaan model saat eksekusi:** `gemini-3.8-flash` mengalami HTTP 429 setelah beberapa percobaan, kemudian sistem berhasil menggunakan model cadangan `gemini-3.6-flash`. Output akhir menunjukkan mode `AI recommendation (gemini-3.6-flash)`, bukan fallback rule-based. Ini menunjukkan mekanisme pergantian model berjalan pada skenario tersebut.

### 🧪 Hasil pengujian yang tercatat

| Pengujian | Hasil run |
|---|---:|
| Validasi katalog dan aturan rekomendasi | **35/35 lulus** |
| Pengujian status, input, dan perhitungan MHM | **11/11 lulus** |
| Pengujian lapisan AI tanpa memanggil API (retry, fallback, ID katalog, tindakan wajib) | **15/15 lulus** |

Total terdapat **61 pemeriksaan otomatis yang lulus** pada tiga kelompok pengujian tersebut. Angka ini adalah hasil pemeriksaan kode pada skenario yang disiapkan, bukan bukti bahwa rekomendasi telah divalidasi secara klinis/teknis atau siap digunakan untuk keputusan nyata.

### ⚠️ Keterbatasan dan tindak lanjut

1. Tinjau acuan 15 liter/orang/hari, tambahan operasional MHM 5 liter/orang/hari, dan ambang status 1 serta 3 hari bersama sumber resmi dan mentor/pakar.
2. Tinjau ulang urutan rekomendasi AI, khususnya posisi tindakan wajib MHM, dan pertimbangkan aturan agar tindakan wajib tampil pada posisi yang mudah terlihat.
3. Uji API Gemini asli pada status Krisis dan Belum Dapat Ditentukan, dengan variasi jumlah anggota dan kebutuhan MHM. Pengujian lapisan AI 15/15 di atas menggunakan respons simulasi, bukan 15 panggilan API.
4. Lengkapi `dasar_rujukan` dengan dokumen dan halaman yang spesifik. Rujukan umum seperti “WHO/UNICEF/CDC; adaptasi AIRA” belum cukup untuk audit.
5. Tambahkan input dugaan kontaminasi atau tentukan secara eksplisit bagaimana aturan wajib kontaminasi diaktifkan.
6. Hubungkan aturan Navigation Air dengan status data titik air yang benar-benar tersedia dan terverifikasi.
7. Lakukan uji pengguna untuk memastikan istilah, satuan, dan pesan rekomendasi mudah dipahami.

### Pemenuhan kriteria assignment

| Kriteria | Bukti |
|---|---|
| Fitur dan problem statement jelas | Kalkulator Ketahanan Air dan sistem rekomendasi (Bagian 1) |
| Pipeline dapat dijalankan | Katalog Supabase berhasil dimuat; perhitungan dan rekomendasi berjalan; Gemini utama 429 lalu model cadangan berhasil |
| Output dan bukti pengujian tersedia | Skenario utama, perbandingan rekomendasi, 35 + 11 + 15 pemeriksaan otomatis |
| Keterbatasan dan next steps dijelaskan | Responsible AI dan tindak lanjut di atas |
| Kode dan dokumentasi | Notebook, modul bersama, requirements, dan README dalam repository |

> Pipeline ini masih prototipe. Hasil kalkulator memperkirakan kuantitas/durasi cadangan, bukan kualitas atau keamanan air untuk diminum.


## 📦 Menjalankan notebook dan menyimpan ke repository

Notebook ini ditujukan untuk Google Colab atau Jupyter. Untuk menjalankan versi Colab:

1. Simpan `SUPABASE_URL` dan `SUPABASE_SERVICE_ROLE_KEY` pada Colab Secrets. `GEMINI_API_KEY` opsional untuk lapisan AI.
2. Jalankan notebook dari atas ke bawah. Pastikan sumber katalog yang tercetak adalah `Supabase (tabel katalog_tindakan)` jika ingin mereproduksi hasil run ini.
3. Jika diminta, unggah `aira_calculator_shared.py` dari folder `src/` repository.
4. Simpan salinan notebook dengan output eksekusi terbaru. Hapus output yang mungkin memuat informasi sensitif sebelum membagikan repository publik.

**Dependensi:** gunakan `requirements-calculator.txt` untuk notebook Kalkulator. `requirements.txt` di root merupakan gabungan dependensi kedua notebook.

**Struktur repository:** lihat `README.md` di root. Karena Kalkulator dan Assistant memakai modul rumus yang sama, keduanya direkomendasikan berada dalam satu repository AIRA (monorepo), bukan menyalin dan memelihara dua versi modul bersama.
